# 1. IDE、图像表示与基础操作

**目标**：能稳定运行 OpenCV-Python，理解图像在计算机中的表示；建立对 OpenCV 库本身的定位认知。

本 Notebook 的**核心原则**：

> **每一处讲解，都配一段可运行代码；每一段代码，都显示结果。**
> 不讲空话，全部用实例说话。

**本 Notebook 的代码组织原则**：

> **每个代码单元都相对独立、可单独运行**，不依赖前面单元的执行结果。
> 需要图像时，单元内自己读、自己兜底，保证从任意一个单元开始都能跑通。

**注释风格**：

> 每一行代码都配有逐行注释，说明这行在做什么、为什么这么写。

**输出风格**：

> 涉及数值运算的地方，**先输出算式，再输出结果**，让读者看清每一步怎么来的。

主要内容：
1. OpenCV 在图像处理中的地位（用实例说明它能做什么）
2. 安装与环境准备（实际检测版本）
3. 版本差异与常见坑（每个坑跑代码验证）
4. 图像在计算机中的表示（读真实图 + 显示 + 结构）
5. 色彩空间基础（同一图转 5 种空间并排显示）
6. 图像的读取、显示、保存（全部实际执行）
7. 视频与摄像头的输入输出（写视频 + 读回 + 显示）
8. 像素访问、ROI、通道分离与合并（可视化每个操作）
9. 图像算术、位运算、掩码（对比 + 三图并排）
10. 性能基础（实测耗时对比）

In [ ]:
# ============ 环境自检（本单元独立，不依赖其他单元） ============
import sys                       # 导入 sys，用于读取 Python 版本信息
import numpy as np               # 导入 NumPy，OpenCV 的底层数组库，约定俗成缩写为 np
import cv2                       # 导入 OpenCV-Python，约定俗成缩写为 cv2
import matplotlib.pyplot as plt  # 导入 Matplotlib 的绘图模块，缩写为 plt，用于在 Notebook 中显示图像

# 让 Matplotlib 在 Notebook 中正常显示中文与负号
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS', 'DejaVu Sans']  # 按优先级选择可用的中文字体
plt.rcParams['axes.unicode_minus'] = False  # 关闭 Unicode 负号，避免负号显示成方块

print('Python :', sys.version.split()[0])     # 打印 Python 版本：取版本字符串的第一段（如 3.11.5）
print('OpenCV :', cv2.__version__)            # 打印 OpenCV 版本号
print('NumPy  :', np.__version__)             # 打印 NumPy 版本号

---
# 1.1 OpenCV 在图像处理中的地位

## 1.1.1 起源与定位

OpenCV（Open Source Computer Vision Library）：

- **C++ 编写**、跨平台、开源；
- 由 **Intel** 发起，现由 OpenCV.org 维护；
- **传统机器视觉领域事实标准库**；
- `OpenCV-Python` 只是它的 **Python 绑定封装**，核心算法依旧跑 C++。

> 你写的 `cv2.xxx()`，大部分是在调用底层 C++ 实现。

## 1.1.2 能力边界

**OpenCV 能做**（都会在后续章节展开）：

- 图像读写、显示、保存；
- 像素操作、滤波、形态学；
- 几何变换（缩放、旋转、仿射、透视）；
- 边缘检测、轮廓提取；
- 特征提取（SIFT、ORB、AKAZE）；
- 相机标定、立体视觉；
- 视频分析（光流、背景建模、跟踪）；
- 深度学习推理（`cv2.dnn`）。

**OpenCV 不做**：

- 深度学习训练（用 PyTorch / TensorFlow）；
- 复杂 GUI 界面（用 Qt、Tkinter）；
- 数据可视化报表（用 Matplotlib、Seaborn）。

## 1.1.3 行业应用场景

- 工业质检：缺陷检测、尺寸测量、定位；
- 安防监控：运动检测、目标跟踪、人脸识别；
- 机器人视觉：抓取定位、SLAM 前端、避障；
- 自动驾驶感知底层：车道线、标定、预处理；
- 相机标定：单目/双目/鱼眼；
- AR 标记识别：ArUco、二维码；
- 嵌入式视觉：ARM、Jetson、树莓派；
- 教学科研：算法验证、快速原型。

工业界传统视觉方案首选 OpenCV。

## 1.1.4 和同类库对比

| 库 | 定位 | 优势 | 局限 |
|----|------|------|------|
| PIL/Pillow | 图像读写、简单像素处理 | 轻量、易用 | 无轮廓、特征、标定等高级视觉算法 |
| scikit-image | Python 原生视觉库 | API 优雅、算法全 | 性能弱、工业落地少 |
| PyTorch/TensorFlow | 深度学习训练+推理 | 自动微分、GPU | 不擅长传统图像预处理、几何变换、相机几何 |
| OpenCV | 传统 CV 工具箱 | 算法成熟、性能高、跨平台 | 高层封装弱、版本 API 有变更 |

## 1.1.5 优缺点总结

**优势**：

- 算法成熟、性能高；
- 跨平台（PC / ARM / 嵌入式）；
- 海量案例、社区庞大；
- 商用免费。

**局限**：

- 高层封装弱，偏向底层算法；
- Python 接口只是封装，纯 Python 循环容易产生性能瓶颈；
- 不同 API 版本间存在接口变更。

## 1.1.6 学习定位

OpenCV 是**计算机视觉的工具箱**，学会它不等于掌握全部 CV，
重点是学会用它实现视觉算法，而不是死记 API。

## 1.1.7 实例：OpenCV 到底能做什么？

光说没用，直接跑代码。下面这段演示**读图 + 灰度转换 + 边缘检测 + 高斯模糊**四个基础操作，
并把结果**并排显示**出来，让读者一眼看到 OpenCV 的实际能力。

In [ ]:
# ============ 实例：OpenCV 四个基础操作（本单元自包含） ============
import os                        # 导入 os，用于判断文件/路径是否存在
import numpy as np               # 导入 NumPy，用于构造兜底测试图
import cv2                       # 导入 OpenCV，用于图像读写与处理
import matplotlib.pyplot as plt  # 导入 Matplotlib，用于在 Notebook 中显示图像

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS', 'DejaVu Sans']  # 设置中文字体优先级
plt.rcParams['axes.unicode_minus'] = False  # 关闭负号 Unicode 化，避免显示异常

# --- 读图：优先真实图，读不到则生成一张测试图 ---
REAL_IMG_PATH = 'P&V/cat.jpg'                            # 定义真实图像路径
img = cv2.imread(REAL_IMG_PATH) if os.path.exists(REAL_IMG_PATH) else None  # 存在则读，否则置 None
if img is None:                                          # 如果没读到（文件不存在或读取失败）
    img = np.zeros((400, 500, 3), dtype=np.uint8)        # 兜底：构造 400×500 的 3 通道全黑图（uint8）
    cv2.rectangle(img, (100, 100), (300, 300), (0, 0, 255), -1)  # 画一个红色实心矩形（BGR）
    cv2.circle(img, (400, 120), 60, (0, 255, 0), -1)     # 画一个绿色实心圆（BGR）
    cv2.putText(img, 'Test', (150, 380), cv2.FONT_HERSHEY_SIMPLEX, 2, (255, 255, 255), 3)  # 写白色文字

gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)             # 灰度转换：BGR 3 通道 → 单通道灰度
blur = cv2.GaussianBlur(gray, (15, 15), 0)               # 高斯模糊：(15,15) 为核大小，0 表示标准差自动计算
edges = cv2.Canny(gray, 100, 200)                        # Canny 边缘检测：低阈值 100，高阈值 200

img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)           # 转 RGB，便于用 Matplotlib 正确显示颜色

plt.figure(figsize=(14, 4))                              # 创建画布，宽 14 高 4 英寸

plt.subplot(1, 4, 1); plt.imshow(img_rgb);          plt.title('Original');      plt.axis('off')  # 第 1 格：显示原图（RGB）
plt.subplot(1, 4, 2); plt.imshow(gray, cmap='gray'); plt.title('Grayscale');     plt.axis('off')  # 第 2 格：灰度图用 gray 色图显示
plt.subplot(1, 4, 3); plt.imshow(blur, cmap='gray'); plt.title('Gaussian Blur'); plt.axis('off')  # 第 3 格：模糊后的灰度图
plt.subplot(1, 4, 4); plt.imshow(edges, cmap='gray');plt.title('Canny Edges');   plt.axis('off')  # 第 4 格：边缘检测结果

plt.tight_layout()                                       # 自动调整子图间距，避免标题重叠
plt.show()                                               # 渲染并显示图像

**观察结果**：

- **Original**：原图；
- **Grayscale**：单通道灰度，信息量减少但结构还在；
- **Gaussian Blur**：模糊后细节被平滑；
- **Canny Edges**：只保留轮廓，白线是边缘。

这就是 OpenCV 最基本的能力：**读图 → 变换 → 显示**。

## 1.1.8 实例：OpenCV vs Pillow vs Matplotlib 读同一张图

三个库读同一张图，**通道顺序不同**：

- OpenCV：BGR；
- Pillow：RGB；
- Matplotlib：RGB。

下面并排显示，让读者看到"如果直接用 OpenCV 读、Matplotlib 显示"，颜色会偏。

In [ ]:
# ============ 实例：三库读取对比（本单元自包含） ============
import os                        # 导入 os，用于判断文件是否存在
import numpy as np               # 导入 NumPy，用于生成兜底测试图
import cv2                       # 导入 OpenCV，用其 imread 读取图像
import matplotlib.pyplot as plt  # 导入 Matplotlib，用于显示图像

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS', 'DejaVu Sans']  # 设置中文字体
plt.rcParams['axes.unicode_minus'] = False  # 关闭负号 Unicode 化

# --- 读图：优先真实图，读不到则生成一张彩色测试图 ---
REAL_IMG_PATH = 'P&V/cat.jpg'                            # 真实图像路径
img_cv = cv2.imread(REAL_IMG_PATH) if os.path.exists(REAL_IMG_PATH) else None  # 存在则读，否则 None
if img_cv is None:                                       # 如果读不到
    img_cv = np.zeros((300, 400, 3), dtype=np.uint8)     # 兜底：构造 300×400 全黑图
    img_cv[:, :, 2] = 255                                # 把第三通道（BGR 下的 R）设为 255，整体呈红色
    cv2.circle(img_cv, (200, 150), 80, (255, 255, 0), -1)  # 画一个青色实心圆

img_cv_rgb = cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB)      # BGR -> RGB，让 Matplotlib 能正确显示

plt.figure(figsize=(10, 5))                               # 创建画布

plt.subplot(1, 2, 1)                                      # 左图
plt.imshow(img_cv)                                        # 直接把 BGR 数据当 RGB 显示，颜色会错
plt.title('OpenCV BGR shown as RGB (wrong color)')        # 标题说明颜色错误
plt.axis('off')                                           # 关闭坐标轴

plt.subplot(1, 2, 2)                                      # 右图
plt.imshow(img_cv_rgb)                                    # 显示已转好的 RGB
plt.title('BGR -> RGB (correct)')                         # 标题说明颜色正确
plt.axis('off')                                           # 关闭坐标轴

plt.tight_layout()                                        # 调整子图间距
plt.show()                                                # 渲染显示

**结论**：左图红蓝互换，右图正常。

> OpenCV 读图默认 BGR，Matplotlib 显示默认 RGB，**必须转换**。

---
# 1.2 安装与环境准备

## 1.2.1 三种安装方式

**pip**（最常用）：

```bash
pip install opencv-python
```

**conda**（Anaconda 用户）：

```bash
conda install -c conda-forge opencv
```

**源码编译**：适合定制、嵌入式，新手不建议。

## 1.2.2 三个 pip 包的区别

| 包名 | 含 contrib | 含 GUI | 适用场景 |
|------|-----------|--------|----------|
| `opencv-python` | ❌ | ✅ | 桌面开发、学习 |
| `opencv-contrib-python` | ✅ | ✅ | SIFT、ArUco、跟踪 |
| `opencv-python-headless` | ❌ | ❌ | 服务器、Docker |

**三个包不能同时安装**。

In [ ]:
# ============ 实例：检测当前环境（本单元自包含） ============
import sys                       # 导入 sys，用于获取 Python 版本
import numpy as np               # 导入 NumPy，用于打印版本号
import cv2                       # 导入 OpenCV，用于打印版本号与优化信息

print('Python :', sys.version.split()[0])                # Python 版本：取版本字符串第一段
print('OpenCV :', cv2.__version__)                       # OpenCV 版本号
print('NumPy  :', np.__version__)                        # NumPy 版本号
print('优化开启:', cv2.useOptimized())                   # 查询 OpenCV 是否启用了内部优化（默认 True）
print('线程数  :', cv2.getNumThreads())                  # 查询 OpenCV 使用的并行线程数

# 检测是否有 contrib（用 SIFT 是否存在判断）
try:                                                      # 尝试执行可能失败的语句
    sift = cv2.SIFT_create()                              # 尝试创建 SIFT 对象；不存在会抛 AttributeError
    print('contrib : 可用（SIFT 存在）')                   # 成功则说明 contrib 可用
except AttributeError:                                    # 如果 SIFT 不存在
    print('contrib : 不可用（SIFT 不存在，需装 opencv-contrib-python）')  # 提示安装 contrib

In [ ]:
# ============ 实例：getBuildInformation 查看构建信息（本单元自包含） ============
import cv2                       # 导入 OpenCV，用于读取构建信息

info = cv2.getBuildInformation()                          # 获取 OpenCV 编译时的完整配置信息（字符串）
print(info[:1200])                                        # 只打印前 1200 字符，避免刷屏

# 快速提取几个关键字段
for key in ['OpenCV version', 'GUI', 'OpenCL', 'IPP', 'Parallel framework']:  # 依次遍历要查找的关键词
    for line in info.splitlines():                        # 把构建信息按行切分后逐行扫描
        if key.lower() in line.lower():                   # 忽略大小写做关键词匹配
            print(line.strip())                           # 打印匹配到的整行（去掉首尾空白）
            break                                         # 找到一条就跳出内层循环，避免重复打印

## 1.2.3 Anaconda 是否自带 OpenCV？

**不一定**：

| 安装方式 | 是否自带 OpenCV |
|----------|------------------|
| Anaconda 早期版本 | 多数自带 |
| Anaconda 较新版本 | **默认不带** |
| Miniconda | **不带** |

**验证**：

```bash
python -c "import cv2; print(cv2.__version__)"
```

没有就装：`conda install -c conda-forge opencv` 或 `pip install opencv-python`。

**建议**：不在 base 堆包，单独建环境：

```bash
conda create -n cv python=3.11
conda activate cv
conda install -c conda-forge opencv
```

---
# 1.3 版本差异与常见坑

## 1.3.1 主要版本

| 版本 | 特点 | 现状 |
|------|------|------|
| 2.x | C 风格 API | 淘汰 |
| 3.x | C++ API，SIFT 移 contrib | 老项目 |
| 4.x | 主流，DNN 增强 | 推荐 |
| 5.x | 部分 API 调整 | 推进中 |

## 1.3.2 常见坑（每个坑跑代码验证）

**坑 1：SIFT 在 contrib**

上面 1.2.2 已经用 `cv2.SIFT_create()` 验证过。

**坑 2：常量名 vs 整数**

- `cv2.IMREAD_COLOR = 1`；
- `cv2.IMREAD_GRAYSCALE = 0`；
- `cv2.IMREAD_UNCHANGED = -1`；
- **建议始终用常量名**。

**坑 3：findContours 返回值**

3.x 返回 3 个值，4.x 返回 2 个值。下面实测。

In [ ]:
# ============ 实例：验证常量值（本单元自包含） ============
import os                        # 导入 os，用于判断文件是否存在
import numpy as np               # 导入 NumPy，用于构造兜底图和比较数组
import cv2                       # 导入 OpenCV，用于读写图像

print('cv2.IMREAD_COLOR     =', cv2.IMREAD_COLOR, ' ← 等价于整数 1')       # 打印彩色读取常量
print('cv2.IMREAD_GRAYSCALE =', cv2.IMREAD_GRAYSCALE, ' ← 等价于整数 0')   # 打印灰度读取常量
print('cv2.IMREAD_UNCHANGED =', cv2.IMREAD_UNCHANGED, ' ← 等价于整数 -1')  # 打印原样读取常量

# --- 读图：优先真实图，读不到则生成一张测试图 ---
REAL_IMG_PATH = 'P&V/cat.jpg'                            # 真实图像路径
if not os.path.exists(REAL_IMG_PATH):                    # 如果真实图不存在
    tmp = np.zeros((100, 100, 3), dtype=np.uint8)        # 兜底：构造 100×100 全黑图
    tmp[:, :, 0] = 200                                   # 把 B 通道置 200，整体偏蓝
    parent = os.path.dirname(REAL_IMG_PATH) or '.'       # 取父目录；若为空则用当前目录
    if os.path.isdir(parent):                            # 父目录存在
        cv2.imwrite(REAL_IMG_PATH, tmp)                  # 直接写入原路径
    else:                                                # 父目录不存在
        REAL_IMG_PATH = '_tmp_test.jpg'                  # 改用当前目录下的临时文件
        cv2.imwrite(REAL_IMG_PATH, tmp)                  # 写入临时文件

img1 = cv2.imread(REAL_IMG_PATH, 0)                       # 用整数 0 读灰度
img2 = cv2.imread(REAL_IMG_PATH, cv2.IMREAD_GRAYSCALE)    # 用常量名读灰度
print('整数 0 读到的 shape  :', img1.shape)               # 打印用整数读到的形状
print('常量读到的 shape    :', img2.shape)               # 打印用常量读到的形状
print('整数读 vs 常量读 是否一致:', np.array_equal(img1, img2))  # 应为 True

In [ ]:
# ============ 实例：findContours 返回值（本单元自包含） ============
import os                        # 导入 os，用于判断文件是否存在
import numpy as np               # 导入 NumPy，用于构造兜底图
import cv2                       # 导入 OpenCV，用于图像处理

# --- 读图：优先真实图，读不到则生成测试图 ---
REAL_IMG_PATH = 'P&V/cat.jpg'                            # 真实图像路径
img = cv2.imread(REAL_IMG_PATH) if os.path.exists(REAL_IMG_PATH) else None  # 存在则读，否则 None
if img is None:                                          # 若读不到
    img = np.zeros((300, 300, 3), dtype=np.uint8)        # 兜底：构造 300×300 全黑图
    cv2.rectangle(img, (80, 80), (220, 220), (255, 255, 255), -1)  # 画白色实心矩形，便于找轮廓

gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)              # 转灰度
_, thresh = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)  # 二值化：>127 置 255，否则 0；_ 忽略返回的阈值

result = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)  # 只取外轮廓，压缩冗余点
print('findContours 返回值个数:', len(result), '（4.x 为 2；3.x 为 3）')  # 打印返回值个数

# 兼容两种版本
if len(result) == 2:                                      # 4.x 返回 (contours, hierarchy)
    contours, hierarchy = result                          # 直接解包
else:                                                     # 3.x 返回 (image, contours, hierarchy)
    _, contours, hierarchy = result                       # 丢弃第一个返回值

print('轮廓数量:', len(contours))                         # 打印检测到的轮廓数量

**坑 4：GUI 后端差异**

- Linux 可能缺 Qt / GTK；
- Jupyter 中 `imshow` 可能弹不出窗口；
- 远程服务器无 GUI，建议用 headless + Matplotlib。

**坑 5：中文路径**

- `cv2.imread` 对中文路径支持不稳定；
- 解决：`cv2.imdecode(np.fromfile(path, np.uint8), flags)`。

---
# 1.4 图像在计算机中的表示

## 1.4.1 图像 = ndarray（先看真实图像）

在 OpenCV-Python 中，图像就是 **NumPy 数组**：

```
shape = (height, width, channels)
dtype = uint8（大多情况）
```

**先读一张真实图，显示出来，再打印结构**，让读者看到：**那张图，就是这个数组**。

In [ ]:
# ============ 实例：真实图像 + 显示 + 结构（本单元自包含） ============
import os                        # 导入 os，用于判断文件是否存在
import numpy as np               # 导入 NumPy，用于生成兜底图
import cv2                       # 导入 OpenCV，用于读写图像
import matplotlib.pyplot as plt  # 导入 Matplotlib，用于显示图像

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS', 'DejaVu Sans']  # 中文字体
plt.rcParams['axes.unicode_minus'] = False  # 关闭负号 Unicode 化

# --- 读图：优先真实图，读不到则生成测试图 ---
REAL_IMG_PATH = 'P&V/cat.jpg'                            # 真实图像路径
img = cv2.imread(REAL_IMG_PATH) if os.path.exists(REAL_IMG_PATH) else None  # 存在则读，否则 None
if img is None:                                          # 若读不到
    img = np.zeros((400, 500, 3), dtype=np.uint8)        # 兜底：构造 400×500 全黑图
    cv2.rectangle(img, (100, 100), (300, 300), (0, 0, 255), -1)  # 画红色实心矩形
    cv2.circle(img, (400, 120), 60, (0, 255, 0), -1)     # 画绿色实心圆

img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)            # BGR -> RGB，便于 Matplotlib 正确显示

# 打印结构
print('=' * 50)                                          # 打印一行分隔线
print('【真实图像】', REAL_IMG_PATH if os.path.exists(REAL_IMG_PATH) else '（兜底测试图）')  # 打印使用的图像路径
print('=' * 50)                                          # 打印一行分隔线
print('shape :', img.shape, ' = (高 H, 宽 W, 通道 C)')    # 形状
print('dtype :', img.dtype)                               # 元素类型，一般 uint8
print('ndim  :', img.ndim, '（彩色图为 3：H、W、C）')      # 维度数
print('size  :', img.size, ' = H × W × C =', img.shape[0], '×', img.shape[1], '×', img.shape[2])  # 元素总数
print('H, W, C =', img.shape[0], img.shape[1], img.shape[2])  # 分别打印高、宽、通道数
print('左上角像素 (B, G, R):', img[0, 0])                 # 左上角像素的 BGR 值
print('中心像素   (B, G, R):', img[img.shape[0]//2, img.shape[1]//2])  # 中心像素的 BGR 值
print('最小/最大像素值:', img.min(), img.max())           # 全图最小和最大像素值

# 显示
plt.figure(figsize=(5, 5))                                # 创建 5×5 英寸画布
plt.imshow(img_rgb)                                       # 显示 RGB 图
plt.title(f'Real image  shape={img.shape}')               # 标题带上图像形状
plt.axis('off')                                           # 关闭坐标轴
plt.show()                                                # 渲染显示

**结论**：

- 屏幕上的图，就是 `img` 这个数组；
- `img.shape = (H, W, 3)` 表示 H 行、W 列、3 通道；
- `img[0, 0]` 就是**左上角**那个像素的 BGR 值；
- **数组里每个数值 = 图像上一个像素**。

## 1.4.2 数组切片 = 图像的一块（ROI 对应）

`img[y1:y2, x1:x2]` 取出的，就是图像上**左上角 `(x1, y1)`、右下角 `(x2, y2)` 的一块**。

下面把原图和 ROI 并排显示，证明"切片 = 取图像一块"。

In [ ]:
# ============ 实例：ROI 对应关系可视化（本单元自包含） ============
import os                        # 导入 os，用于判断文件是否存在
import numpy as np               # 导入 NumPy，用于生成兜底图
import cv2                       # 导入 OpenCV，用于图像读写和绘图
import matplotlib.pyplot as plt  # 导入 Matplotlib，用于显示

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS', 'DejaVu Sans']  # 中文字体
plt.rcParams['axes.unicode_minus'] = False  # 关闭负号 Unicode 化

# --- 读图：优先真实图，读不到则生成测试图 ---
REAL_IMG_PATH = 'P&V/cat.jpg'                            # 真实图像路径
img = cv2.imread(REAL_IMG_PATH) if os.path.exists(REAL_IMG_PATH) else None  # 存在则读，否则 None
if img is None:                                          # 若读不到
    img = np.zeros((400, 500, 3), dtype=np.uint8)        # 兜底图
    cv2.rectangle(img, (100, 100), (300, 300), (0, 0, 255), -1)  # 红色实心矩形
    cv2.circle(img, (400, 120), 60, (0, 255, 0), -1)     # 绿色实心圆

img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)            # BGR -> RGB

y1, y2, x1, x2 = 100, 250, 150, 350          # ROI 边界：行起始、行结束、列起始、列结束
y2 = min(y2, img.shape[0]); x2 = min(x2, img.shape[1])    # 防止边界超出图像尺寸
roi = img[y1:y2, x1:x2]                      # 按 [行, 列] 切片取 ROI
roi_rgb = cv2.cvtColor(roi, cv2.COLOR_BGR2RGB)  # ROI 转 RGB

img_boxed = img_rgb.copy()                   # 拷贝一份原图，避免污染显示
cv2.rectangle(img_boxed, (x1, y1), (x2, y2), (255, 0, 0), 3)  # 在拷贝图上画蓝色矩形框，线宽 3

plt.figure(figsize=(10, 5))                  # 画布
plt.subplot(1, 2, 1)                         # 左图
plt.imshow(img_boxed)                        # 显示带框原图
plt.title(f'Full image  {img.shape}')        # 标题带尺寸
plt.axis('off')                              # 关闭坐标轴
plt.subplot(1, 2, 2)                         # 右图
plt.imshow(roi_rgb)                          # 显示 ROI 内容
plt.title(f'ROI  img[{y1}:{y2}, {x1}:{x2}]  shape={roi.shape}')  # 标题带上切片表达式和形状
plt.axis('off')                              # 关闭坐标轴
plt.tight_layout()                           # 调整间距
plt.show()                                   # 渲染

print('ROI shape:', roi.shape, ' = (', y2 - y1, ',', x2 - x1, ', 3)')  # 打印 ROI 形状 = (行数, 列数, 通道)
print('ROI 左上角像素 (B, G, R):', roi[0, 0])  # ROI 左上角像素
print('原图对应位置像素     :', img[y1, x1])   # 验证 roi[0,0] == img[y1,x1]
print('两者是否相等         :', np.array_equal(roi[0, 0], img[y1, x1]))  # True

**结论**：

- `img[y1:y2, x1:x2]` 和原图框选区域**内容完全一致**；
- `roi[0, 0] == img[y1, x1]`；
- ROI 是**视图**，改 ROI 会影响原图。

## 1.4.3 视图（view）vs 副本（copy）

### 一句话核心

> **`img[100:200, 100:200]` 取出来的不是一块“新图”，而是原数组的一个“窗口”。**
> 你通过这个窗口改数据，改的是**原数组本身**。

### 生活比喻

把原图想成一面墙，ROI 切片是你在这面墙上**开的一扇窗**：

- **视图（view）**：窗户。透过窗户看到的还是那面墙，你在窗户上刷漆，墙也变了；
- **副本（copy）**：拍照。你把墙拍成一张照片，在照片上涂改，墙不受影响。

### 为什么必须强调

这是**高频 bug 来源**：

1. 你只想改 ROI，结果原图被改了：
   ```python
   roi = img[100:200, 100:200]
   roi[:] = 0        # 你以为只清了 ROI，其实原图那块也黑了
   ```
2. 你以为在做“抠图”，其实只是开了个窗：
   ```python
   face = img[50:150, 100:200]   # 这不是“抠出来的脸”
   # 它和原图共享内存，原图一改，face 也变
   ```
3. 面试/考试常考：NumPy 切片默认是视图，`copy()` 才是副本。

### 判断方法

```python
roi.flags.owndata      # False → 视图；True → 副本
roi.base is img        # True  → 是 img 的视图
```

### 一句话总结

> **切片 = 开窗（视图），`copy()` = 拍照（副本）。**
> 想独立操作、不影响原图，必须 `.copy()`。

下面用代码验证。

In [ ]:
# ============ 实例：视图 vs 副本（本单元自包含） ============
import os                        # 导入 os，用于判断文件是否存在
import numpy as np               # 导入 NumPy，用于生成兜底图
import cv2                       # 导入 OpenCV，用于读写图像

# --- 读图：优先真实图，读不到则生成测试图 ---
REAL_IMG_PATH = 'P&V/cat.jpg'                            # 真实图像路径
img = cv2.imread(REAL_IMG_PATH) if os.path.exists(REAL_IMG_PATH) else None  # 存在则读，否则 None
if img is None:                                          # 若读不到
    img = np.zeros((400, 500, 3), dtype=np.uint8)        # 兜底图
    cv2.rectangle(img, (100, 100), (300, 300), (0, 0, 255), -1)  # 红色实心矩形

img2 = img.copy()                                     # 先拷贝一份，避免污染原始 img
roi_view = img2[100:200, 100:200]                     # 切片：这是一个视图（共享内存）
roi_copy = img2[100:200, 100:200].copy()              # 显式拷贝：这是一个独立的副本

print('roi_view 是否拥有数据:', roi_view.flags.owndata, ' ← False 表示是视图')   # 视图不拥有数据
print('roi_copy 是否拥有数据:', roi_copy.flags.owndata, ' ← True 表示是副本')    # 副本拥有独立数据
print('roi_view.base is img2:', roi_view.base is img2, ' ← True 表示共享 img2 内存')  # 视图的 base 指向原数组
print('roi_copy.base is img2:', roi_copy.base is img2, ' ← False 表示独立')          # 副本不是原数组的视图

before = img2[100, 100].copy()                        # 记下视图改动前的像素
roi_view[:] = [0, 0, 255]                             # 把视图整块赋值成红色（BGR）
after_view = img2[100, 100].copy()                    # 记下视图改动后的像素
print(f'改视图：img2[100,100] {before} -> {after_view}，证明改动传导到原图')  # 对比变化

before = img2[100, 100].copy()                        # 记下副本改动前的像素
roi_copy[:] = [0, 255, 0]                             # 把副本整块赋值成绿色
after_copy = img2[100, 100].copy()                    # 记下副本改动后的像素
print(f'改副本：img2[100,100] {before} -> {after_copy}，证明副本不影响原图')  # 对比变化

## 1.4.4 把图像缩小，看清"像素 = 方块"

把真实图缩到 **6×6**，关闭插值（`interpolation='nearest'`），每个像素显示成一个方块。

In [ ]:
# ============ 实例：缩小到 6x6 显示（本单元自包含） ============
import os                        # 导入 os，用于判断文件是否存在
import numpy as np               # 导入 NumPy，用于生成兜底图
import cv2                       # 导入 OpenCV，用于读取图像并缩放
import matplotlib.pyplot as plt  # 导入 Matplotlib，用于显示

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS', 'DejaVu Sans']  # 中文字体
plt.rcParams['axes.unicode_minus'] = False  # 关闭负号 Unicode 化

# --- 读图：优先真实图，读不到则生成测试图 ---
REAL_IMG_PATH = 'P&V/cat.jpg'                            # 真实图像路径
img = cv2.imread(REAL_IMG_PATH) if os.path.exists(REAL_IMG_PATH) else None  # 存在则读，否则 None
if img is None:                                          # 若读不到
    img = np.zeros((400, 500, 3), dtype=np.uint8)        # 兜底图
    cv2.rectangle(img, (100, 100), (300, 300), (0, 0, 255), -1)  # 红色实心矩形

print('缩放前 shape :', img.shape, f' = (H={img.shape[0]}, W={img.shape[1]}, C=3)')  # 打印缩放前形状
small = cv2.resize(img, (6, 6), interpolation=cv2.INTER_AREA)  # 用区域平均法缩到 6×6
print('缩放后 shape :', small.shape, ' = (H=6, W=6, C=3)')   # 打印缩放后形状
small_rgb = cv2.cvtColor(small, cv2.COLOR_BGR2RGB)             # BGR -> RGB

plt.figure(figsize=(5, 5))                                # 创建 5×5 画布
plt.imshow(small_rgb, interpolation='nearest')            # 关闭插值，每个像素显示成一个方块
plt.title(f'Downscaled to 6x6  shape={small.shape}')      # 标题带上形状
plt.axis('off')                                           # 关闭坐标轴
plt.show()                                                # 渲染

print('缩小后 dtype:', small.dtype)                       # uint8
print('缩小后左上角像素 BGR:', small[0, 0])               # 左上角像素 BGR
print('缩小后完整数组：')
print(small)                                              # 打印整张 6×6×3 数组，方便对照屏幕

## 1.4.5 手动构造小数组，也显示出来

手动构造一张 **4×6 的蓝色图**，把数组和图像的对应关系**缩小到肉眼可数**。

In [ ]:
# ============ 实例：手动构造 4x6 蓝色图（本单元自包含） ============
import numpy as np               # 导入 NumPy，用于构造数组
import cv2                       # 导入 OpenCV，用于颜色空间转换
import matplotlib.pyplot as plt  # 导入 Matplotlib，用于显示

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS', 'DejaVu Sans']  # 中文字体
plt.rcParams['axes.unicode_minus'] = False  # 关闭负号 Unicode 化

img_small = np.zeros((4, 6, 3), dtype=np.uint8)  # 构造 4 行 6 列 3 通道全 0 的 BGR 图
img_small[:, :, 0] = 255                         # 把 B 通道全部置 255，整体呈蓝色
img_small_rgb = cv2.cvtColor(img_small, cv2.COLOR_BGR2RGB)  # BGR -> RGB

plt.figure(figsize=(3, 3))                       # 创建小画布
plt.imshow(img_small_rgb, interpolation='nearest')  # 关闭插值，每个像素显示为方块
plt.title(f'Manual 4x6 blue  shape={img_small.shape}')  # 标题带形状
plt.axis('off')                                  # 关闭坐标轴
plt.show()                                       # 渲染

print('shape :', img_small.shape, ' = (H=4, W=6, C=3)')  # 形状
print('dtype :', img_small.dtype)                # uint8
print('ndim  :', img_small.ndim)                 # 3
print('size  :', img_small.size, ' = 4 × 6 × 3 = 72')  # 元素总数，带算式
print('像素(0,0):', img_small[0, 0], ' ← 这是 (B=255, G=0, R=0)，所以在 BGR 里显示为蓝色')  # 左上角像素 BGR
print('整个数组：')
print(img_small)                                 # 打印整个数组，方便对照

## 1.4.6 shape / dtype / ndim / size

| 属性 | 含义 | 示例 |
|------|------|------|
| `shape` | 各维度大小 | `(414, 500, 3)` |
| `dtype` | 元素类型 | `uint8` |
| `ndim` | 维度数 | `3` |
| `size` | 元素总数 | `414*500*3` |

- 取高：`img.shape[0]`；
- 取宽：`img.shape[1]`；
- 取通道：`img.shape[2]`；
- 判灰度：`img.ndim == 2`。

## 1.4.7 BGR vs RGB

- OpenCV 默认 **BGR**；
- Matplotlib / PIL 默认 **RGB**；
- 混用颜色会偏。

转换：

```python
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
img_rgb = img[:, :, ::-1]
```

## 1.4.8 像素值的含义

- `uint8`：0~255；
- `uint16`：0~65535；
- `float32`：0.0~1.0 或 0~255.0；
- 超范围会溢出或截断。

## 1.4.9 8位 / 16位 / 32F 图像的差异与适用场景

| dtype | 范围 | 典型来源 | 适用场景 |
|-------|------|----------|----------|
| `uint8` | 0~255 | 普通图片、JPEG/PNG | 显示、常规处理 |
| `uint16` | 0~65535 | 医学影像、深度图、RAW | 高动态范围、精确计算 |
| `float32` | 0.0~1.0 或 0~255.0 | 滤波中间结果、DNN | 数值计算、避免溢出 |

**注意**：

- `imshow` 显示 `float32` 时，Matplotlib 会按 0~1 显示，若值是 0~255 会全白；
- `cv2` 很多函数要求 `uint8` 或 `float32`，输入 `uint16` 可能报错；
- 转换用 `img.astype(np.float32)` 或 `cv2.normalize`。

下面实际构造并观察。

In [ ]:
# ============ 实例：8位 / 16位 / 32F 对比（本单元自包含） ============
import os                        # 导入 os，用于判断文件是否存在
import numpy as np               # 导入 NumPy，用于构造不同 dtype 的数组
import cv2                       # 导入 OpenCV，用于读写图像和颜色转换
import matplotlib.pyplot as plt  # 导入 Matplotlib，用于显示

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS', 'DejaVu Sans']  # 中文字体
plt.rcParams['axes.unicode_minus'] = False  # 关闭负号 Unicode 化

img_u8 = np.array([[0, 128, 255]], dtype=np.uint8)          # 8 位示例：0、128、255
img_u16 = np.array([[0, 32768, 65535]], dtype=np.uint16)    # 16 位示例：0、32768、65535
img_f32 = np.array([[0.0, 0.5, 1.0]], dtype=np.float32)     # 32F 示例：0.0、0.5、1.0

print('uint8   :', img_u8, ' dtype=', img_u8.dtype,   ' 范围 = 0 ~ 2^8 - 1 = 0~255')    # 8 位
print('uint16  :', img_u16, ' dtype=', img_u16.dtype, ' 范围 = 0 ~ 2^16 - 1 = 0~65535') # 16 位
print('float32 :', img_f32, ' dtype=', img_f32.dtype, ' 范围 = 0.0 ~ 1.0 或 0~255.0')    # 32F

# --- 读图：优先真实图，读不到则生成测试图 ---
REAL_IMG_PATH = 'P&V/cat.jpg'                            # 真实图像路径
img = cv2.imread(REAL_IMG_PATH) if os.path.exists(REAL_IMG_PATH) else None  # 存在则读，否则 None
if img is None:                                          # 若读不到
    img = np.zeros((300, 400, 3), dtype=np.uint8)        # 兜底图
    cv2.rectangle(img, (80, 80), (320, 220), (0, 0, 255), -1)  # 红色实心矩形

# 把真实图转成不同 dtype，观察差异
img_f = img.astype(np.float32) / 255.0                      # 先转 float32，再除以 255 归一化到 0~1
print('原图 uint8 范围   :', img.min(), '~', img.max(), '   = 0~255')          # 原图范围
print('转 float32 范围   :', img_f.min(), '~', img_f.max(), ' = uint8 / 255 后归一化到 0.0~1.0')  # 归一化范围

plt.figure(figsize=(10, 4))                                 # 画布
plt.subplot(1, 2, 1)                                        # 左图
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))            # uint8 图像正常显示
plt.title('uint8 (0~255)')                                  # 标题
plt.axis('off')                                             # 关闭坐标轴
plt.subplot(1, 2, 2)                                        # 右图
plt.imshow(cv2.cvtColor(img_f, cv2.COLOR_BGR2RGB))          # float32 图像（0~1）正常显示
plt.title('float32 (0~1)')                                  # 标题
plt.axis('off')                                             # 关闭坐标轴
plt.tight_layout()                                          # 调整间距
plt.show()                                                  # 渲染

## 1.4.10 图像内存布局：行优先、连续内存、步长（stride）

- NumPy 数组默认 **行优先（C order）**：一行一行连续存储；
- `img.strides` 表示每个维度跳多少字节；
- `img.flags['C_CONTIGUOUS']` 判断是否连续；
- ROI 切片可能不连续，`cv2` 某些函数要求连续内存，可用 `np.ascontiguousarray` 转换。

下面查看真实图的内存布局。

In [ ]:
# ============ 实例：内存布局与步长（本单元自包含） ============
import os                        # 导入 os，用于判断文件是否存在
import numpy as np               # 导入 NumPy，用于生成兜底图和转连续数组
import cv2                       # 导入 OpenCV，用于读取图像

# --- 读图：优先真实图，读不到则生成测试图 ---
REAL_IMG_PATH = 'P&V/cat.jpg'                            # 真实图像路径
img = cv2.imread(REAL_IMG_PATH) if os.path.exists(REAL_IMG_PATH) else None  # 存在则读，否则 None
if img is None:                                          # 若读不到
    img = np.zeros((400, 500, 3), dtype=np.uint8)        # 兜底图
    cv2.rectangle(img, (100, 100), (300, 300), (0, 0, 255), -1)  # 红色实心矩形

H, W, C = img.shape                                       # 分别取高、宽、通道数
print('img shape     :', img.shape)                           # (H, W, 3)
print('img dtype     :', img.dtype, ' ← 每元素 1 字节（uint8）')  # uint8
print('img strides   :', img.strides)                         # 每跳到下一行/下一列/下一通道所需字节数
print(f'  一行字节数    = W × C = {W} × {C} = {W * C} 字节')   # 一行字节数，带算式
print(f'  一个像素字节数= C     = {C} 字节')                   # 一个像素字节数
print('C_CONTIGUOUS  :', img.flags['C_CONTIGUOUS'], ' ← 是否行优先连续，一般 True')   # 行优先连续
print('F_CONTIGUOUS  :', img.flags['F_CONTIGUOUS'], ' ← 是否列优先连续，一般 False')  # 列优先连续

# ROI 切片是否连续
roi = img[100:200, 100:200]                                   # 取一块 ROI（视图，行间有间隙，通常不连续）
print('ROI C_CONTIGUOUS:', roi.flags['C_CONTIGUOUS'], ' ← 切片后一般 False，因为一行只取一段')  # 可能 False
roi_cont = np.ascontiguousarray(roi)                          # 把它复制成内存连续的数组
print('转连续后 C_CONTIGUOUS:', roi_cont.flags['C_CONTIGUOUS'], ' ← True，已单独复制一份连续内存')  # True

---
# 1.5 色彩空间基础

## 1.5.1 常见色彩空间

| 色彩空间 | 通道 | 用途 |
|----------|------|------|
| 灰度 | 1 | 亮度，简化计算 |
| BGR | 3 | OpenCV 默认 |
| RGB | 3 | 显示、通用 |
| HSV | 3 | 颜色分割、色调分析 |
| YCrCb | 3 | 视频压缩、肤色检测 |
| Lab | 3 | 颜色差异度量 |

## 1.5.2 转换函数

```python
cv2.cvtColor(src, code)  # cvt：convert
```

常用 code：`COLOR_BGR2GRAY`、`COLOR_BGR2RGB`、`COLOR_BGR2HSV`、`COLOR_BGR2YCrCb`、`COLOR_BGR2Lab`。

In [ ]:
# ============ 实例：同一图转 5 种色彩空间并排显示（本单元自包含） ============
import os                        # 导入 os，用于判断文件是否存在
import numpy as np               # 导入 NumPy，用于生成兜底图
import cv2                       # 导入 OpenCV，用于颜色空间转换
import matplotlib.pyplot as plt  # 导入 Matplotlib，用于显示

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS', 'DejaVu Sans']  # 中文字体
plt.rcParams['axes.unicode_minus'] = False  # 关闭负号 Unicode 化

# --- 读图：优先真实图，读不到则生成测试图 ---
REAL_IMG_PATH = 'P&V/cat.jpg'                            # 真实图像路径
img = cv2.imread(REAL_IMG_PATH) if os.path.exists(REAL_IMG_PATH) else None  # 存在则读，否则 None
if img is None:                                          # 若读不到
    img = np.zeros((300, 400, 3), dtype=np.uint8)        # 兜底图
    cv2.rectangle(img, (80, 80), (320, 220), (0, 0, 255), -1)  # 红色实心矩形
    cv2.circle(img, (200, 150), 60, (0, 255, 0), -1)     # 绿色实心圆

gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)       # 灰度：单通道
rgb  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)        # RGB：3 通道
hsv  = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)        # HSV：3 通道
ycrcb = cv2.cvtColor(img, cv2.COLOR_BGR2YCrCb)     # YCrCb：3 通道
lab  = cv2.cvtColor(img, cv2.COLOR_BGR2Lab)        # Lab：3 通道

print('各色彩空间 shape:')                            # 打印各数组形状，验证通道数
print('  BGR   :', img.shape,   ' ← 3 通道')          # 原图
print('  Gray  :', gray.shape,  ' ← 1 通道（无 C 维）')  # 灰度
print('  RGB   :', rgb.shape,   ' ← 3 通道')          # RGB
print('  HSV   :', hsv.shape,   ' ← 3 通道')          # HSV
print('  YCrCb :', ycrcb.shape, ' ← 3 通道')          # YCrCb
print('  Lab   :', lab.shape,   ' ← 3 通道')          # Lab

print('左上角像素值对比（同一物理位置，不同表示）：')  # 对比同一位置在不同色彩空间下的表示
print('  BGR   :', img[0, 0],   ' ← (B, G, R)')       # 原图 BGR
print('  Gray  :', gray[0, 0],  ' ← 单通道亮度')       # 灰度值
print('  HSV   :', hsv[0, 0],   ' ← (H, S, V)')       # HSV 三元组
print('  YCrCb :', ycrcb[0, 0], ' ← (Y, Cr, Cb)')     # YCrCb 三元组
print('  Lab   :', lab[0, 0],   ' ← (L, a, b)')       # Lab 三元组

plt.figure(figsize=(15, 8))                         # 画布
plt.subplot(2, 3, 1); plt.imshow(rgb);               plt.title('RGB');           plt.axis('off')  # RGB 正常显示
plt.subplot(2, 3, 2); plt.imshow(gray, cmap='gray'); plt.title('Gray');          plt.axis('off')  # 灰度
plt.subplot(2, 3, 3); plt.imshow(hsv);               plt.title('HSV');           plt.axis('off')  # HSV 仅形状示意
plt.subplot(2, 3, 4); plt.imshow(ycrcb);             plt.title('YCrCb');         plt.axis('off')  # YCrCb 仅形状示意
plt.subplot(2, 3, 5); plt.imshow(lab);               plt.title('Lab');           plt.axis('off')  # Lab 仅形状示意
plt.subplot(2, 3, 6); plt.imshow(img[:, :, ::-1]);   plt.title('BGR(as RGB)');   plt.axis('off')  # 倒序通道得到 RGB
plt.tight_layout()                                  # 调整间距
plt.show()                                          # 渲染

**注意**：HSV / YCrCb / Lab 直接 `imshow` 显示只是形状示意，颜色会怪，
因为它们的三通道不是 RGB 语义。

## 1.5.3 HSV 为什么常用

- **H（Hue）**：色调，0~179（OpenCV）；
- **S（Saturation）**：饱和度，0~255；
- **V（Value）**：明度，0~255。

**光照变化时 H 相对稳定**，做颜色分割比 BGR 更稳。

---
# 1.6 图像的读取、显示、保存

## 1.6.1 读取 cv2.imread

```python
img = cv2.imread(filename, flags)
```

常用 flags：

- `cv2.IMREAD_COLOR`（默认）：3 通道 BGR；
- `cv2.IMREAD_GRAYSCALE`：单通道灰度；
- `cv2.IMREAD_UNCHANGED`：保留 alpha。

读取失败返回 `None`，**必须判断**。

In [ ]:
# ============ 实例：三种 flags 读取对比（本单元自包含） ============
import os                        # 导入 os，用于判断文件是否存在
import numpy as np               # 导入 NumPy，用于生成兜底图
import cv2                       # 导入 OpenCV，用于 imread 读图
import matplotlib.pyplot as plt  # 导入 Matplotlib，用于显示

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS', 'DejaVu Sans']  # 中文字体
plt.rcParams['axes.unicode_minus'] = False  # 关闭负号 Unicode 化

# --- 准备一张可读的图（真实图优先，否则生成并落盘） ---
REAL_IMG_PATH = 'P&V/cat.jpg'                             # 真实图像路径
if not os.path.exists(REAL_IMG_PATH):                     # 若真实图不存在
    tmp = np.zeros((300, 400, 3), dtype=np.uint8)         # 兜底：构造全黑图
    cv2.rectangle(tmp, (80, 80), (320, 220), (0, 0, 255), -1)  # 红色实心矩形
    cv2.circle(tmp, (200, 150), 60, (0, 255, 0), -1)      # 绿色实心圆
    REAL_IMG_PATH = '_tmp_flags.jpg'                      # 改用当前目录下的临时文件
    cv2.imwrite(REAL_IMG_PATH, tmp)                       # 写入临时文件，保证后面可读

img_color = cv2.imread(REAL_IMG_PATH, cv2.IMREAD_COLOR)      # 彩色：3 通道 BGR
img_gray = cv2.imread(REAL_IMG_PATH, cv2.IMREAD_GRAYSCALE)   # 灰度：1 通道
img_unch = cv2.imread(REAL_IMG_PATH, cv2.IMREAD_UNCHANGED)   # 原样：保留 alpha（这里无 alpha，等同彩色）

print('COLOR     shape:', img_color.shape, ' ← (H, W, 3)，3 通道')  # 彩色
print('GRAYSCALE shape:', img_gray.shape, ' ← (H, W)，无通道维')   # 灰度
print('UNCHANGED shape:', img_unch.shape, ' ← (H, W, 3) 或 (H, W, 4)')  # 原样

plt.figure(figsize=(12, 4))                     # 画布
plt.subplot(1, 3, 1); plt.imshow(cv2.cvtColor(img_color, cv2.COLOR_BGR2RGB)); plt.title('COLOR'); plt.axis('off')  # 彩色
plt.subplot(1, 3, 2); plt.imshow(img_gray, cmap='gray'); plt.title('GRAYSCALE'); plt.axis('off')  # 灰度
plt.subplot(1, 3, 3); plt.imshow(cv2.cvtColor(img_unch[:, :, :3], cv2.COLOR_BGR2RGB)); plt.title('UNCHANGED (first 3 ch)'); plt.axis('off')  # 原样前 3 通道
plt.tight_layout()                              # 调整间距
plt.show()                                      # 渲染

bad = cv2.imread('not_exist.jpg')               # 故意读一个不存在的文件
print('读取不存在的文件 not_exist.jpg 返回值:', bad, ' ← None 表示读取失败')  # 应为 None

## 1.6.2 显示 cv2.imshow

```python
cv2.imshow(winname, img)
cv2.waitKey(0)
cv2.destroyAllWindows()
```

- `imshow` 不阻塞；
- `waitKey` 驱动事件循环；
- `destroyAllWindows` 清理。

Jupyter 中弹窗可能不稳定，用 try 保护。

In [ ]:
# ============ 实例：OpenCV 窗口显示（无 GUI 自动跳过；本单元自包含） ============
import os                        # 导入 os，用于判断文件是否存在
import numpy as np               # 导入 NumPy，用于生成兜底图
import cv2                       # 导入 OpenCV，用于窗口显示

# --- 读图：优先真实图，读不到则生成测试图 ---
REAL_IMG_PATH = 'P&V/cat.jpg'                            # 真实图像路径
img_color = cv2.imread(REAL_IMG_PATH) if os.path.exists(REAL_IMG_PATH) else None  # 存在则读，否则 None
if img_color is None:                                    # 若读不到
    img_color = np.zeros((300, 400, 3), dtype=np.uint8)  # 兜底图
    cv2.rectangle(img_color, (80, 80), (320, 220), (0, 0, 255), -1)  # 红色实心矩形

def cv_show(name, image):                    # 封装显示函数：名字 + 图像
    cv2.imshow(name, image)                  # 弹出窗口显示图像（不阻塞）
    cv2.waitKey(0)                           # 等待任意按键（0 表示无限等待）
    cv2.destroyAllWindows()                  # 关闭所有 OpenCV 窗口

try:                                         # 用 try 保护：无 GUI 环境会抛异常
    cv_show('cat', img_color)                # 调用显示函数
    print('OpenCV 窗口已显示')                # 成功提示
except Exception as e:                       # 捕获异常
    print('当前环境无法弹窗，跳过。原因:', e)  # 打印失败原因

## 1.6.3 保存 cv2.imwrite

```python
cv2.imwrite(filename, img)
```

- 成功 True，失败 False；
- 格式由扩展名决定；
- 可传质量 / 压缩参数。

In [ ]:
# ============ 实例：保存 + 读回验证（本单元自包含） ============
import os                        # 导入 os，用于判断文件是否存在
import numpy as np               # 导入 NumPy，用于生成兜底图和比较数组
import cv2                       # 导入 OpenCV，用于读写图像

# --- 读图：优先真实图，读不到则生成测试图 ---
REAL_IMG_PATH = 'P&V/cat.jpg'                            # 真实图像路径
img_color = cv2.imread(REAL_IMG_PATH) if os.path.exists(REAL_IMG_PATH) else None  # 存在则读，否则 None
if img_color is None:                                    # 若读不到
    img_color = np.zeros((300, 400, 3), dtype=np.uint8)  # 兜底图
    cv2.rectangle(img_color, (80, 80), (320, 220), (0, 0, 255), -1)  # 红色实心矩形

ok1 = cv2.imwrite('out_cat.png', img_color)                        # 保存为 PNG（无损）
ok2 = cv2.imwrite('out_cat.jpg', img_color,                        # 保存为 JPEG
                  [cv2.IMWRITE_JPEG_QUALITY, 90])                   # 指定 JPEG 质量为 90（1~100）
print('PNG 保存返回 :', ok1, ' ← True 表示成功')                    # True
print('JPG 保存返回 :', ok2, ' ← True 表示成功')                    # True

back = cv2.imread('out_cat.png')                                   # 读回 PNG
print('读回 shape   :', back.shape, ' 原图 shape:', img_color.shape, ' → 一致')  # 形状应一致
print('读回是否等于原图:', np.array_equal(back, img_color), ' ← PNG 无损，True')  # True

## 1.6.4 中文路径问题

### 现象：`imread` / `imwrite` 遇到中文路径可能失败

- **Windows 上大概率失败**：`cv2.imread` 返回 `None`，`cv2.imwrite` 返回 `False`；
- **Linux / macOS 上通常正常**：系统 locale 是 UTF-8，`fopen` 能直接处理；
- 因此这是**平台相关**的问题，不是“一定失败”。

### 原因：OpenCV 底层用 C 的 `fopen`，不保证认 UTF-8

- OpenCV 核心是 **C++**，`imread` / `imwrite` 底层调用 C 标准库的 `fopen`；
- `fopen` 在 **Windows** 上按 **本地编码（GBK）** 解释路径；
- Python 3 的字符串是 **UTF-8**；
- 两者对不上 → OpenCV 拿到乱码路径 → 找不到文件。

> 一句话：**`imread` / `imwrite` 对中文路径的支持是平台相关的，不能保证跨平台可用。**

### 解法：绕开 `imread` / `imwrite`，自己读字节流

- 读：`np.fromfile(path)` 读字节 → `cv2.imdecode(buf, flag)` 解码成图像；
- 写：`cv2.imencode(ext, img)` 编码成字节 → `.tofile(path)` 写文件。

Python 的文件接口能正确处理 UTF-8 路径，OpenCV 只负责编解码，**与路径无关**。

### 什么时候必须这么写？

| 场景 | 是否需要 imdecode / imencode |
|------|------------------------------|
| 路径全英文 / 数字 | ❌ 直接用 `imread` / `imwrite` |
| 路径含中文（Windows） | ✅ 必须绕开 |
| 路径含中文（Linux / macOS） | 视 locale 而定，建议绕开 |
| 路径含 emoji / 日文 / 韩文 | ✅ 建议绕开 |

> 保守做法：**只要路径可能含非 ASCII 字符，就统一用 `imdecode` / `imencode`。**

### 下面的代码会实测

1. 先用 `imwrite` / `imread` **直接尝试**中文路径，打印真实结果；
2. 再用 `imencode` / `imdecode` **跨平台写法**读写，并验证内容一致。

In [ ]:
# ============ 实例：中文路径读写（本单元自包含） ============
import os                        # 导入 os，用于判断文件是否存在
import numpy as np               # 导入 NumPy，用于生成兜底图和比较数组
import cv2                       # 导入 OpenCV，用于读写图像
import matplotlib.pyplot as plt  # 导入 Matplotlib，用于显示

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS', 'DejaVu Sans']  # 中文字体
plt.rcParams['axes.unicode_minus'] = False  # 关闭负号 Unicode 化

# --- 读图：优先真实图，读不到则生成测试图 ---
REAL_IMG_PATH = 'P&V/cat.jpg'                            # 真实图像路径
img_color = cv2.imread(REAL_IMG_PATH) if os.path.exists(REAL_IMG_PATH) else None  # 存在则读，否则 None
if img_color is None:                                    # 若读不到
    img_color = np.zeros((300, 400, 3), dtype=np.uint8)  # 兜底图
    cv2.rectangle(img_color, (80, 80), (320, 220), (0, 0, 255), -1)  # 红色实心矩形

cn_path = '中文测试图.png'                                          # 使用含中文的文件名

# --- 第一步：先测当前环境下，直接用 imread / imwrite 处理中文路径行不行 ---
ok_direct = cv2.imwrite(cn_path, img_color)                        # 直接写中文路径
print('cv2.imwrite 到中文路径 :', ok_direct, ' ← True 表示当前环境支持')  # Windows 常为 False

bad_direct = cv2.imread(cn_path)                                   # 直接读中文路径
if bad_direct is None:                                             # 若读不到
    print('cv2.imread 从中文路径读: None  ← 当前环境不支持，必须绕开')
else:                                                              # 若读到了
    print('cv2.imread 从中文路径读: shape =', bad_direct.shape,
          ' ← 当前环境支持，但仍建议统一用 imdecode / imencode 保证跨平台')

# --- 第二步：不管上面结果如何，统一用跨平台写法 ---
cv2.imencode('.png', img_color)[1].tofile(cn_path)                 # 编码为 PNG 字节流后写入文件（Python 文件接口处理 UTF-8 路径）
print('已保存到中文路径:', cn_path)                                # 提示保存成功

loaded = cv2.imdecode(np.fromfile(cn_path, dtype=np.uint8),        # 读字节流
                      cv2.IMREAD_COLOR)                            # 再解码成 BGR 图像
print('读回 shape:', loaded.shape, ' 原图 shape:', img_color.shape, ' → 一致')  # 形状应与原图一致

# --- 第三步：验证读回的内容和原图一致 ---
print('读回是否等于原图:', np.array_equal(loaded, img_color), ' ← PNG 无损，True')  # True

# --- 第四步：显示，确认图像正常 ---
plt.figure(figsize=(5, 5))                                         # 画布
plt.imshow(cv2.cvtColor(loaded, cv2.COLOR_BGR2RGB))                # 显示读回的图像
plt.title('Loaded from Chinese path')                              # 标题
plt.axis('off')                                                    # 关闭坐标轴
plt.show()                                                         # 渲染

## 1.6.5 图像元数据（EXIF）读取基础

**EXIF**（Exchangeable Image File Format，可交换图像文件格式），
是**嵌在图片文件里的一组元数据**，主要由数码相机、手机拍照时自动写入。

**它记录了什么？**

| 类别 | 典型字段 | 说明 |
|------|----------|------|
| 拍摄时间 | `DateTime`、`DateTimeOriginal` | 拍这张照片的时间 |
| 相机信息 | `Make`、`Model` | 品牌、型号（如 Apple、iPhone 15） |
| 曝光参数 | `ExposureTime`、`FNumber`、`ISOSpeedRatings` | 快门、光圈、ISO |
| 焦距 | `FocalLength` | 镜头焦距 |
| 方向 | `Orientation` | 图像旋转方向（1~8） |
| GPS | `GPSInfo` | 经纬度、海拔 |
| 软件 | `Software` | 处理/导出软件 |

**为什么有用？**

- **图像管理**：按拍摄时间、地点自动归档；
- **隐私风险**：EXIF 里可能含 GPS 坐标、设备型号，发图前要清除；
- **算法处理**：`Orientation` 决定图像是否需要旋转，否则可能"躺着"；
- **取证 / 溯源**：判断图片是否被 PS 过。

**关键事实**：

- **不是所有图都有 EXIF**：截图、网图、经过某些软件另存后可能被剥离；
- **PNG 一般没有 EXIF**（个别支持），**JPEG / TIFF 最常见**；
- **OpenCV 基本读不到 EXIF**，要用 **Pillow** 或 **exifread**。

### 下面的代码做什么

1. 先读一张图，**打印是否含 EXIF**；
2. 如果有，**逐字段打印**；
3. 如果没有，**明确说明原因**（不是代码错，是图本身没有）；
4. 最后**手写一张带 EXIF 的 JPEG**（用 Pillow 保存时写入 EXIF），再读回来，**确保能真实读到**。

In [ ]:
# ============ 实例：用 Pillow 读取 EXIF（本单元自包含） ============
import os                        # 导入 os，用于判断文件是否存在
import numpy as np               # 导入 NumPy，用于生成兜底图
import cv2                       # 导入 OpenCV，用 imread 对比 EXIF 情况

try:                                                  # 尝试导入 Pillow
    from PIL import Image, ExifTags                   # Image 打开图片；ExifTags 提供标签名映射
except ImportError:                                   # Pillow 未安装
    print('未安装 Pillow，跳过 EXIF。可 pip install Pillow')
else:                                                 # Pillow 已安装
    REAL_IMG_PATH = 'P&V/Switch.jpg'                  # 真实图像路径
    if not os.path.exists(REAL_IMG_PATH):             # 若真实图不存在
        tmp = np.zeros((300, 400, 3), dtype=np.uint8) # 兜底图
        cv2.rectangle(tmp, (80, 80), (320, 220), (0, 0, 255), -1)  # 红色实心矩形
        REAL_IMG_PATH = '_tmp_exif.jpg'               # 改用当前目录下的临时文件
        cv2.imwrite(REAL_IMG_PATH, tmp)               # 落盘，保证 Pillow 能打开

    img_path = REAL_IMG_PATH                          # 打开路径
    pil_img = Image.open(img_path)                    # 用 Pillow 打开图像
    exif = pil_img.getexif()                          # 提取 EXIF 字典（可能为空）
    print('=' * 50)                                   # 分隔线
    print('图像:', img_path)                          # 打印图像路径
    print('格式:', pil_img.format, ' 尺寸:', pil_img.size)  # 打印格式和尺寸
    print('EXIF 条目数:', len(exif), ' ← 0 表示没有 EXIF')  # 0 表示没有 EXIF
    print('=' * 50)                                   # 分隔线

    if exif:                                          # 如果存在 EXIF
        print('EXIF 内容：')                          # 提示
        for tag_id, value in exif.items():            # 遍历所有标签
            tag = ExifTags.TAGS.get(tag_id, tag_id)   # 把数字标签转为可读名；没有就保留数字
            print(f'  {tag:25s}: {value}')            # 按固定宽度打印
    else:                                             # 如果没有 EXIF
        print('该图没有 EXIF 信息。')                  # 说明原因
        print('原因：截图 / 网图 / 被编辑软件剥离，或本身就是 PNG。')

    print('\n' + '=' * 50)                           # 分隔线
    print('下面手动写入 EXIF，再读回，确保代码真的能读到：')
    print('=' * 50)                                   # 分隔线

    test_exif = Image.Exif()                          # 新建一个 EXIF 容器
    test_exif[271] = 'TestMake'                       # 271 = Make 厂商
    test_exif[272] = 'TestModel'                      # 272 = Model 型号
    test_exif[306] = '2024:01:01 12:00:00'            # 306 = DateTime 拍摄时间
    test_exif[274] = 1                                # 274 = Orientation 方向

    pil_img.save('exif_demo.jpg', exif=test_exif)     # 另存为 JPEG 并写入 EXIF

    back = Image.open('exif_demo.jpg')                # 重新打开刚才保存的图
    back_exif = back.getexif()                        # 读取其 EXIF
    print('读回 EXIF 条目数:', len(back_exif), ' ← 大于 0，证明代码能读到')  # 应 > 0
    for tag_id, value in back_exif.items():           # 逐条打印
        tag = ExifTags.TAGS.get(tag_id, tag_id)       # 标签名
        print(f'  {tag:25s}: {value}')                # 打印标签和值

    img_cv = cv2.imread('exif_demo.jpg')              # 用 OpenCV 读同一张图
    print('\nOpenCV 读到的 shape:', img_cv.shape, ' ← 能读图，但 OpenCV 没有 EXIF 接口')

---
# 1.7 视频与摄像头的输入输出

## 1.7.1 VideoCapture

```python
cap = cv2.VideoCapture(source)
```

- `0` 默认摄像头；
- 或视频文件路径；
- `cap.read()` 返回 `(ret, frame)`；
- 用完 `cap.release()`。

In [ ]:
# ============ 实例：尝试用 DSHOW 打开摄像头（本单元自包含） ============
import time                      # 导入 time，用于 sleep 给设备初始化时间
import cv2                       # 导入 OpenCV，用于摄像头操作

print("尝试使用 DSHOW 打开摄像头...")              # 提示开始尝试
cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)          # 打开 0 号摄像头，并强制使用 DirectShow 后端（Windows 常用）

time.sleep(1)                                     # 等待 1 秒，给设备足够时间初始化

if not cap.isOpened():                            # 如果打不开
    print("❌ 使用 DSHOW 依然无法打开摄像头。")    # 打印失败信息
else:                                             # 如果成功打开
    print("✅ 成功打开！尝试读取画面...")          # 打印成功信息
    while True:                                   # 无限循环，逐帧读取
        ret, frame = cap.read()                   # 读一帧；ret 表示是否成功，frame 是图像
        if not ret:                               # 读取失败
            print("❌ 读不到画面")                 # 打印提示
            break                                 # 退出循环
        cv2.imshow("Camera Test (Press Q to quit)", frame)  # 显示当前帧
        if cv2.waitKey(30) & 0xFF == ord('q'):    # 等待 30ms；按 q 键退出
            break                                 # 退出循环
    cap.release()                                 # 释放摄像头
    cv2.destroyAllWindows()                       # 关闭所有窗口

## 1.7.2 视频读取属性：get(CAP_PROP_*) / set(CAP_PROP_*)

常用属性：

- `CAP_PROP_FRAME_WIDTH`：帧宽；
- `CAP_PROP_FRAME_HEIGHT`：帧高；
- `CAP_PROP_FPS`：帧率；
- `CAP_PROP_FRAME_COUNT`：总帧数；
- `CAP_PROP_POS_FRAMES`：当前帧位置。

`set` 可跳帧或改分辨率（摄像头）。

In [ ]:
# ============ 实例：打开摄像头并实时显示（本单元自包含） ============
import cv2                       # 导入 OpenCV，用于摄像头操作

cap = cv2.VideoCapture(0)        # 打开默认摄像头（索引 0）

if not cap.isOpened():           # 若打开失败
    print("无法打开摄像头！")     # 打印提示，不退出内核
else:                            # 若打开成功
    print("摄像头已打开，按 'q' 键退出...")  # 提示操作
    while True:                          # 无限循环读帧
        ret, frame = cap.read()          # 读一帧
        if not ret:                      # 读取失败
            print("无法读取摄像头画面")   # 打印提示
            break                        # 退出循环
        cv2.imshow('Real-time Camera (Press q to quit)', frame)  # 实时显示
        if cv2.waitKey(1) & 0xFF == ord('q'):  # 等待 1ms；按 q 退出
            break                        # 退出循环
    cap.release()                        # 释放摄像头
    cv2.destroyAllWindows()              # 关闭所有窗口

## 1.7.3 VideoWriter

```python
writer = cv2.VideoWriter(filename, fourcc, fps, frameSize)
writer.write(frame)
writer.release()
```

- `fourcc`：4 字符编码；
- `frameSize`：`(width, height)`；
- 写完必须 `release()`。

In [ ]:
# ============ 实例：写 50 帧视频，再读回第一帧显示（本单元自包含） ============
import numpy as np               # 导入 NumPy，用于构造每帧图像
import cv2                       # 导入 OpenCV，用于视频读写
import matplotlib.pyplot as plt  # 导入 Matplotlib，用于显示第一帧

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS', 'DejaVu Sans']  # 中文字体
plt.rcParams['axes.unicode_minus'] = False  # 关闭负号 Unicode 化

fourcc = cv2.VideoWriter_fourcc(*'mp4v')                          # 4 字符编码：mp4v，即 MPEG-4 Part 2
writer = cv2.VideoWriter('demo_out.mp4', fourcc, 20, (320, 240))  # 参数：文件名、编码、帧率 20fps、帧尺寸 (宽, 高)
if writer.isOpened():                                             # 判断写器是否打开成功
    for i in range(50):                                           # 写 50 帧
        frame = np.full((240, 320, 3), i * 5, dtype=np.uint8)     # 构造单色帧，随 i 增大而变亮
        cv2.putText(frame, f'Frame {i}', (20, 120),               # 在帧上写编号
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 255), 2)  # 字体、大小、颜色、线宽
        writer.write(frame)                                       # 把这一帧写入视频
    writer.release()                                              # 必须释放，否则文件可能不完整
    print('已写入 demo_out.mp4，共 50 帧，帧率 20fps，时长 50/20 = 2.5 秒')  # 提示写入完成，顺带算时长
else:                                                             # 如果写器打不开
    print('VideoWriter 打开失败')                                 # 打印提示

cap = cv2.VideoCapture('demo_out.mp4')                            # 打开刚才写的视频
if cap.isOpened():                                                # 判断是否打开成功
    fps = cap.get(cv2.CAP_PROP_FPS)                               # 读取帧率
    n   = cap.get(cv2.CAP_PROP_FRAME_COUNT)                       # 读取总帧数
    w   = cap.get(cv2.CAP_PROP_FRAME_WIDTH)                       # 读取帧宽
    h   = cap.get(cv2.CAP_PROP_FRAME_HEIGHT)                      # 读取帧高
    print(f'读回视频属性：宽={w:.0f}, 高={h:.0f}, FPS={fps:.1f}, 总帧数={n:.0f}, 时长={n/fps:.2f} 秒')  # 打印属性带算式
    ret, frame = cap.read()                                       # 读第一帧
    if ret:                                                       # 若读取成功
        plt.figure(figsize=(4, 3))                                # 画布
        plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))        # 显示第一帧（转 RGB）
        plt.title('First frame of demo_out.mp4')                  # 标题
        plt.axis('off')                                           # 关闭坐标轴
        plt.show()                                                # 渲染
    cap.release()                                                 # 释放视频对象

## 1.7.4 播放视频的典型循环

```python
cap = cv2.VideoCapture('test.mp4')
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    cv2.imshow('frame', frame)
    if cv2.waitKey(25) & 0xFF == ord('q'):
        break
cap.release()
cv2.destroyAllWindows()
```

- `waitKey(25)` 控制速度；
- `& 0xFF` 兼容平台；
- 按 `q` 退出。

## 1.7.5 图像序列的输入输出

图像序列：一组按编号排列的图片，如 `frame_001.png`、`frame_002.png`。

- 读：`cv2.imread(f'frame_{i:03d}.png')`；
- 写：`cv2.imwrite(f'out_{i:03d}.png', frame)`；
- 也可用 `VideoCapture` 读 `frame_%03d.png`（部分后端支持）。

下面演示生成 5 张序列图并读回。

In [ ]:
# ============ 实例：图像序列读写（本单元自包含） ============
import os                        # 导入 os，用于路径拼接和创建目录
import numpy as np               # 导入 NumPy，用于构造每帧图像
import cv2                       # 导入 OpenCV，用于图像读写
import matplotlib.pyplot as plt  # 导入 Matplotlib，用于显示序列

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS', 'DejaVu Sans']  # 中文字体
plt.rcParams['axes.unicode_minus'] = False  # 关闭负号 Unicode 化

seq_dir = 'seq_demo'                                      # 序列目录名
os.makedirs(seq_dir, exist_ok=True)                       # 创建目录，已存在则不报错

for i in range(5):                                        # 生成 5 张
    frame = np.full((60, 80, 3), i * 50, dtype=np.uint8)  # 单色帧，亮度随 i 增加（i=0→0, i=1→50, ...）
    cv2.imwrite(os.path.join(seq_dir, f'frame_{i:03d}.png'), frame)  # 保存为 frame_000.png ... frame_004.png

print('已生成 5 张序列图，亮度依次为：', [i * 50 for i in range(5)])  # 提示生成完成，并列出每张亮度

frames = []                                               # 存放读回的帧
for i in range(5):                                        # 读回 5 张
    path = os.path.join(seq_dir, f'frame_{i:03d}.png')    # 路径拼接
    f = cv2.imread(path)                                  # 读取
    frames.append(f)                                      # 加入列表
    print(f'frame_{i:03d}.png shape:', f.shape, ' 左上角像素 BGR:', f[0, 0])  # 打印形状和左上角像素

plt.figure(figsize=(10, 2))                               # 画布
for i, f in enumerate(frames):                            # 遍历读回的帧
    plt.subplot(1, 5, i + 1)                              # 第 i+1 个子图
    plt.imshow(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))        # 显示（转 RGB）
    plt.title(f'{i} (亮度 {i*50})')                       # 标题用序号 + 亮度
    plt.axis('off')                                       # 关闭坐标轴
plt.tight_layout()                                        # 调整间距
plt.show()                                                # 渲染

---
# 1.8 像素访问、ROI、通道分离与合并

## 1.8.1 像素访问（图上标红点）

- 灰度：`img[y, x]`；
- 彩色：`img[y, x]` 返回 `[B, G, R]`；
- 单通道：`img[y, x, 0]` 取 B。

下面访问几个像素，并在图上**用红点标出**。

In [ ]:
# ============ 实例：像素访问 + 红点标记（本单元自包含） ============
import os                        # 导入 os，用于判断文件是否存在
import numpy as np               # 导入 NumPy，用于生成兜底图
import cv2                       # 导入 OpenCV，用于读取图像和画点
import matplotlib.pyplot as plt  # 导入 Matplotlib，用于显示

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS', 'DejaVu Sans']  # 中文字体
plt.rcParams['axes.unicode_minus'] = False  # 关闭负号 Unicode 化

# --- 读图：优先真实图，读不到则生成测试图 ---
REAL_IMG_PATH = 'P&V/cat.jpg'                            # 真实图像路径
img = cv2.imread(REAL_IMG_PATH) if os.path.exists(REAL_IMG_PATH) else None  # 存在则读，否则 None
if img is None:                                          # 若读不到
    img = np.zeros((400, 500, 3), dtype=np.uint8)        # 兜底图
    cv2.rectangle(img, (100, 100), (300, 300), (0, 0, 255), -1)  # 红色实心矩形

img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)            # BGR -> RGB

pts = [(0, 0),                                        # 左上角 (y, x)
       (img.shape[0]//2, img.shape[1]//2),            # 中心 (y, x)
       (100, 200)]                                    # 自定义点 (y, x)

for (y, x) in pts:                                    # 遍历点
    pixel = img[y, x]                                 # 取出该点像素 BGR
    print(f'img[y={y}, x={x}] = (B={pixel[0]}, G={pixel[1]}, R={pixel[2]})')  # 打印带标签的 BGR

vis = img_rgb.copy()                                  # 拷贝一份用于可视化
for (y, x) in pts:                                    # 遍历点
    cv2.circle(vis, (x, y), 8, (255, 0, 0), -1)       # 在 (x, y) 处画半径 8 的实心红点（RGB 语义下）

plt.figure(figsize=(5, 5))                            # 画布
plt.imshow(vis)                                       # 显示
plt.title('Visited pixels (red dots)')                # 标题
plt.axis('off')                                       # 关闭坐标轴
plt.show()                                            # 渲染

## 1.8.2 ROI（改 ROI 影响原图）

ROI（Region Of Interest） 是视图，不是拷贝。下面演示：取 ROI → 改成绿色 → 显示原图，验证原图也被改。

In [ ]:
# ============ 实例：ROI 修改影响原图（本单元自包含） ============
import os                        # 导入 os，用于判断文件是否存在
import numpy as np               # 导入 NumPy，用于生成兜底图
import cv2                       # 导入 OpenCV，用于读写图像
import matplotlib.pyplot as plt  # 导入 Matplotlib，用于显示

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS', 'DejaVu Sans']  # 中文字体
plt.rcParams['axes.unicode_minus'] = False  # 关闭负号 Unicode 化

# --- 读图：优先真实图，读不到则生成测试图 ---
REAL_IMG_PATH = 'P&V/cat.jpg'                            # 真实图像路径
img = cv2.imread(REAL_IMG_PATH) if os.path.exists(REAL_IMG_PATH) else None  # 存在则读，否则 None
if img is None:                                          # 若读不到
    img = np.zeros((400, 500, 3), dtype=np.uint8)        # 兜底图
    cv2.rectangle(img, (100, 100), (300, 300), (0, 0, 255), -1)  # 红色实心矩形

img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)            # BGR -> RGB（供显示用）

img_mod = img.copy()                                  # 先拷贝一份，避免影响 img 本身
before = img_mod[100, 150].copy()                     # 记录 ROI 修改前该位置的像素
roi = img_mod[100:250, 150:350]                       # 取 ROI：视图，共享内存
roi[:] = [0, 255, 0]                                  # 把 ROI 整块赋值为绿色（BGR）
after = img_mod[100, 150].copy()                      # 记录 ROI 修改后该位置的像素

plt.figure(figsize=(10, 5))                           # 画布
plt.subplot(1, 2, 1)                                  # 左图
plt.imshow(img_rgb)                                   # 原图（未修改）
plt.title('Original')                                 # 标题
plt.axis('off')                                       # 关闭坐标轴
plt.subplot(1, 2, 2)                                  # 右图
plt.imshow(cv2.cvtColor(img_mod, cv2.COLOR_BGR2RGB))  # 修改后的图
plt.title('After ROI = green')                        # 标题
plt.axis('off')                                       # 关闭坐标轴
plt.tight_layout()                                    # 调整间距
plt.show()                                            # 渲染

print(f'img_mod[100,150] 修改前 = {before}，修改后 = {after}')  # 打印改动前后
print('→ 因为 ROI 是视图，改动传导到 img_mod 本身')             # 说明视图特性

## 1.8.3 通道分离与合并

```python
b, g, r = cv2.split(img)
img = cv2.merge([b, g, r])
```

下面把三通道**分别显示**。

In [ ]:
# ============ 实例：三通道分别显示（本单元自包含） ============
import os                        # 导入 os，用于判断文件是否存在
import numpy as np               # 导入 NumPy，用于生成兜底图和比较
import cv2                       # 导入 OpenCV，用于 split/merge
import matplotlib.pyplot as plt  # 导入 Matplotlib，用于显示

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS', 'DejaVu Sans']  # 中文字体
plt.rcParams['axes.unicode_minus'] = False  # 关闭负号 Unicode 化

# --- 读图：优先真实图，读不到则生成测试图 ---
REAL_IMG_PATH = 'P&V/cat.jpg'                            # 真实图像路径
img = cv2.imread(REAL_IMG_PATH) if os.path.exists(REAL_IMG_PATH) else None  # 存在则读，否则 None
if img is None:                                          # 若读不到
    img = np.zeros((300, 400, 3), dtype=np.uint8)        # 兜底图
    img[:, :200, 2] = 255                                # 左半设红（BGR 下 R 在第三通道）
    img[:, 200:, 1] = 255                                # 右半设绿

b, g, r = cv2.split(img)                              # 把 3 通道拆成 3 个单通道数组

print('split 后各通道 shape:', b.shape, g.shape, r.shape, ' ← 都是 (H, W)，没有第 3 维')  # 打印各通道形状

plt.figure(figsize=(12, 4))                           # 画布
plt.subplot(1, 3, 1); plt.imshow(b, cmap='gray'); plt.title('Blue');  plt.axis('off')  # 蓝通道
plt.subplot(1, 3, 2); plt.imshow(g, cmap='gray'); plt.title('Green'); plt.axis('off')  # 绿通道
plt.subplot(1, 3, 3); plt.imshow(r, cmap='gray'); plt.title('Red');   plt.axis('off')  # 红通道
plt.tight_layout()                                    # 调整间距
plt.show()                                            # 渲染

merged = cv2.merge([b, g, r])                         # 按 B、G、R 顺序重新合并
print('合并后 shape:', merged.shape, ' 原图 shape:', img.shape, ' → 一致')  # 应与原图一致
print('合并后是否等于原图:', np.array_equal(merged, img), ' ← True')  # True

---
# 1.9 图像算术、位运算、掩码

## 1.9.1 算术运算：NumPy vs OpenCV

### 先说结论

| 运算 | 库 | 超范围时的行为 | 例：250 + 10 |
|------|----|----------------|--------------|
| `+` | NumPy | **溢出绕回**（mod 256） | 260 mod 256 = **4** |
| `cv2.add` | OpenCV | **饱和截断**（clip 到 255） | min(260, 255) = **255** |
| `-` | NumPy | 溢出绕回（mod 256） | -240 mod 256 = **16** |
| `cv2.subtract` | OpenCV | 饱和截断（clip 到 0） | max(-240, 0) = **0** |

### 为什么？

- NumPy 把 `uint8` 当成**固定位宽的整数**，超出就**截断高位**；
- OpenCV 把图像像素当成**物理量**（亮度、颜色强度），超出就**饱和到边界**；
- 处理图像时，通常 **OpenCV 更符合直觉**：加亮不会突然变暗、减暗不会突然变亮。

下面用代码实测，**每一行都打印算式和结果**。

In [ ]:
# ============ 实例：饱和截断 vs 溢出绕回（本单元自包含） ============
import os                        # 导入 os，用于判断文件是否存在
import numpy as np               # 导入 NumPy，用于加法对比和生成兜底图
import cv2                       # 导入 OpenCV，用于 cv2.add 等

def to_scalar(x):
    """把 numpy 标量 / 任意形状数组 安全转成 Python int（取第一个元素）。"""
    return int(np.asarray(x).reshape(-1)[0])           # 先拉平，再取第 0 个

# ----------------------------------------------------------------
# 一、标量对比：用 shape=(1,1) 的二维数组，兼容 OpenCV 5.0
# ----------------------------------------------------------------
a = np.array([[250]], dtype=np.uint8)                 # shape = (1, 1)
b = np.array([[10]],  dtype=np.uint8)                 # shape = (1, 1)

a_int, b_int = to_scalar(a), to_scalar(b)             # 250, 10

print('=' * 60)
print('标量对比：uint8 只能存 0~255，超范围会怎样？')
print('=' * 60)

# --- 加法 ---
print(f'\n[加法] {a_int} + {b_int} = {a_int + b_int}，超出上限 255')
print(f'  NumPy 加法  :  {a_int} + {b_int} = {a_int + b_int} → mod 256 = {(a_int + b_int) % 256}   ← 溢出绕回')
print(f'  OpenCV 加法 :  {a_int} + {b_int} = {a_int + b_int} → min(…, 255) = 255     ← 饱和截断')

# --- 减法 ---
print(f'\n[减法] {b_int} - {a_int} = {b_int - a_int}，低于下限 0')
print(f'  NumPy 减法  :  {b_int} - {a_int} = {b_int - a_int} → mod 256 = {(b_int - a_int) % 256}    ← 溢出绕回')
print(f'  OpenCV 减法 :  {b_int} - {a_int} = {b_int - a_int} → max(…, 0) = 0        ← 饱和截断')

print(f'\n[正常范围] 250 - 10 = 240，两种方式结果一致：')
print(f'  NumPy 减法  :  {a_int} - {b_int} = {a_int - b_int}')
print(f'  OpenCV 减法 :  {a_int} - {b_int} = {to_scalar(cv2.subtract(a, b))}')

# ----------------------------------------------------------------
# 二、图像整体加 50：对比两种实现，并验证等价
# ----------------------------------------------------------------
print('\n' + '=' * 60)
print('图像整体 +50：OpenCV 饱和 vs NumPy 手动 clip')
print('=' * 60)

# --- 读图：优先真实图，读不到则生成测试图 ---
REAL_IMG_PATH = 'P&V/cat.jpg'                            # 真实图像路径
img = cv2.imread(REAL_IMG_PATH) if os.path.exists(REAL_IMG_PATH) else None  # 存在则读，否则 None
if img is None:                                          # 若读不到
    img = np.zeros((200, 300, 3), dtype=np.uint8)        # 兜底图
    img[:, :, 0] = 100                                   # B 通道统一为 100
    print('未读到真实图，使用兜底图 shape =', img.shape)

bright_cv = cv2.add(img, 50)                          # OpenCV 加：饱和
bright_np = np.clip(img.astype(np.int16) + 50, 0, 255).astype(np.uint8)  # 先转 int16 避免溢出，再 clip 回 uint8

# --- 打印一个具体像素的推导过程 ---
y, x = 0, 0                                            # 取左上角像素
pix = img[y, x]                                        # 原像素 BGR
print(f'\n以像素 img[{y}, {x}] = (B={pix[0]}, G={pix[1]}, R={pix[2]}) 为例：')
for c, name in enumerate(['B', 'G', 'R']):             # 遍历三通道
    v = int(pix[c])                                    # 原值（先转 int，避免 uint8 运算）
    s = v + 50                                         # 数学结果
    print(f'  {name} 通道: 原值 {v} + 50 = {s}  '
          f'→ OpenCV 取 min({s}, 255) = {int(bright_cv[y, x, c])}；'
          f'NumPy clip 后 = {int(bright_np[y, x, c])}')

print('\n两种实现是否完全一致:', np.array_equal(bright_cv, bright_np), ' ← True 表示 OpenCV.add 与 clip 等价')

## 1.9.2 乘法与除法

### 先说结论

| 运算 | 库 | 超范围时的行为 | 例：100 × 3 |
|------|----|----------------|--------------|
| `*` | NumPy | 溢出绕回（mod 256） | 300 mod 256 = **44** |
| `cv2.multiply` | OpenCV | 饱和截断 | min(300, 255) = **255** |

- `cv2.multiply` 常用于**提亮**（乘 >1）、**压暗**（乘 <1）；
- `cv2.divide` 常用于**归一化**；
- 除以 0 时 OpenCV 会返回 0 或饱和值，**不会崩溃**。

下面实测，**打印算式**。

In [ ]:
# ============ 实例：乘法与除法（本单元自包含） ============
import os                        # 导入 os，用于判断文件是否存在
import numpy as np               # 导入 NumPy，用于乘法对比和兜底图
import cv2                       # 导入 OpenCV，用于 multiply / divide
import matplotlib.pyplot as plt  # 导入 Matplotlib，用于显示

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS', 'DejaVu Sans']  # 中文字体
plt.rcParams['axes.unicode_minus'] = False  # 关闭负号 Unicode 化

def to_scalar(x):
    """把 numpy 数组 / 标量 安全转成 Python int（取第一个元素）。"""
    return int(np.asarray(x).reshape(-1)[0])           # 先拉平，再取第 0 个

# ----------------------------------------------------------------
# 一、标量对比：两边都用 (1,1) 的 uint8 数组，兼容 OpenCV 5.0
# ----------------------------------------------------------------
a = np.array([[100]], dtype=np.uint8)                  # shape = (1, 1)
k = np.array([[3]],   dtype=np.uint8)                  # shape = (1, 1)，uint8 才会饱和

print('=' * 60)
print('标量乘法：100 × 3 = 300，超出 uint8 上限 255')
print('=' * 60)
print(f'  NumPy 乘法  :  100 × 3 = 300 → mod 256 = {to_scalar(a * 3)}   ← 溢出绕回')
print(f'  OpenCV 乘法 :  100 × 3 = 300 → min(…, 255) = {to_scalar(cv2.multiply(a, k))}   ← 饱和截断（两边都 uint8 数组）')
print(f'  OpenCV 乘法 :  100 × 3 = 300 → {to_scalar(cv2.multiply(a, 3))}  ← 标量 3（Python int），结果 dtype 是 float64，不饱和')

# --- 除法也顺便测一下 ---
print(f'  OpenCV 除法 :  100 ÷ 3 = {to_scalar(cv2.divide(a, k))}   ← 整数除法')

# ----------------------------------------------------------------
# 二、读图：优先真实图，读不到则生成测试图
# ----------------------------------------------------------------
REAL_IMG_PATH = 'P&V/cat.jpg'                            # 真实图像路径
img = cv2.imread(REAL_IMG_PATH) if os.path.exists(REAL_IMG_PATH) else None  # 存在则读，否则 None
if img is None:                                          # 若读不到
    img = np.zeros((200, 300, 3), dtype=np.uint8)        # 兜底图
    img[:, :, 1] = 120                                   # G 通道统一为 120

img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)        # 供显示用

# ----------------------------------------------------------------
# 三、图像整体乘 1.5：multiply 返回 float64，要 clip 后转回 uint8
# ----------------------------------------------------------------
bright_mul = cv2.multiply(img, 1.5)                   # img 是 uint8，1.5 是 Python float → 结果 dtype=float64
print(f'\ncv2.multiply(img, 1.5) 结果 dtype = {bright_mul.dtype}，'
      f'范围 = {bright_mul.min():.1f} ~ {bright_mul.max():.1f}')
bright_mul = np.clip(bright_mul, 0, 255).astype(np.uint8)  # 先 clip 到 0~255，再转 uint8

# ----------------------------------------------------------------
# 四、打印第一个像素的推导过程（带算式）
# ----------------------------------------------------------------
pix = img[0, 0]                                        # 原像素 BGR
print(f'\n以像素 img[0,0] = (B={pix[0]}, G={pix[1]}, R={pix[2]}) 为例：')
for c, name in enumerate(['B', 'G', 'R']):             # 遍历三通道
    v = int(pix[c])                                    # 原值
    s = v * 1.5                                        # 数学结果
    print(f'  {name} 通道: {v} × 1.5 = {s}  → min({s}, 255) = {int(bright_mul[0,0,c])}')

# ----------------------------------------------------------------
# 五、显示对比
# ----------------------------------------------------------------
plt.figure(figsize=(10, 5))                           # 画布
plt.subplot(1, 2, 1); plt.imshow(img_rgb);            plt.title('Original'); plt.axis('off')  # 原图
plt.subplot(1, 2, 2); plt.imshow(cv2.cvtColor(bright_mul, cv2.COLOR_BGR2RGB)); plt.title('cv2.multiply 1.5'); plt.axis('off')  # 提亮后
plt.tight_layout()                                    # 调整间距
plt.show()                                            # 渲染

## 1.9.3 位运算与掩码（三图并排）

```python
cv2.bitwise_and / or / xor / not
```

构造矩形掩码，只在掩码区域保留原图。

In [ ]:
# ============ 实例：掩码抠图三图并排（本单元自包含） ============
import os                        # 导入 os，用于判断文件是否存在
import numpy as np               # 导入 NumPy，用于构造掩码和兜底图
import cv2                       # 导入 OpenCV，用于位运算
import matplotlib.pyplot as plt  # 导入 Matplotlib，用于显示

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS', 'DejaVu Sans']  # 中文字体
plt.rcParams['axes.unicode_minus'] = False  # 关闭负号 Unicode 化

# --- 读图：优先真实图，读不到则生成测试图 ---
REAL_IMG_PATH = 'P&V/cat.jpg'                            # 真实图像路径
img = cv2.imread(REAL_IMG_PATH) if os.path.exists(REAL_IMG_PATH) else None  # 存在则读，否则 None
if img is None:                                          # 若读不到
    img = np.zeros((400, 500, 3), dtype=np.uint8)        # 兜底图
    cv2.rectangle(img, (100, 100), (300, 300), (0, 0, 255), -1)  # 红色实心矩形

img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)            # 供显示用

mask = np.zeros(img.shape[:2], dtype=np.uint8)        # 构造与原图等宽高的单通道掩码，全 0
mask[100:300, 100:300] = 255                          # 中间矩形区域置 255，表示“参与”
masked = cv2.bitwise_and(img, img, mask=mask)         # 掩码外清零，掩码内保留原值

plt.figure(figsize=(12, 4))                           # 画布
plt.subplot(1, 3, 1); plt.imshow(img_rgb);              plt.title('Original'); plt.axis('off')  # 原图
plt.subplot(1, 3, 2); plt.imshow(mask, cmap='gray');    plt.title('Mask');     plt.axis('off')  # 掩码
plt.subplot(1, 3, 3); plt.imshow(cv2.cvtColor(masked, cv2.COLOR_BGR2RGB)); plt.title('Masked'); plt.axis('off')  # 抠图结果
plt.tight_layout()                                    # 调整间距
plt.show()                                            # 渲染

print('掩码外像素 masked[0, 0] =', masked[0, 0], ' ← 应为 [0, 0, 0]')  # 应为 [0,0,0]
print('掩码内像素 masked[200, 200] =', masked[200, 200], ' 原图 img[200, 200] =', img[200, 200],
      ' → 是否相等:', np.array_equal(masked[200, 200], img[200, 200]))  # 掩码内保留原值

## 1.9.4 掩码的作用

掩码是单通道 `uint8`：

- `0`：不参与；
- `255`：参与。

用途：抠图、局部处理、分割筛选。

## 1.9.5 copyTo 配合 mask

Python 里没有 `copyTo`，对应写法：

- `dst = cv2.bitwise_and(src, src, mask=mask)`：掩码外为 0；
- 或 `dst[mask == 255] = src[mask == 255]`：只复制掩码区域。

下面演示把掩码区域复制到另一张图。

In [ ]:
# ============ 实例：copyTo 对应写法（本单元自包含） ============
import os                        # 导入 os，用于判断文件是否存在
import numpy as np               # 导入 NumPy，用于构造掩码和全黑目标图
import cv2                       # 导入 OpenCV，用于读取图像
import matplotlib.pyplot as plt  # 导入 Matplotlib，用于显示

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS', 'DejaVu Sans']  # 中文字体
plt.rcParams['axes.unicode_minus'] = False  # 关闭负号 Unicode 化

# --- 读图：优先真实图，读不到则生成测试图 ---
REAL_IMG_PATH = 'P&V/cat.jpg'                            # 真实图像路径
img = cv2.imread(REAL_IMG_PATH) if os.path.exists(REAL_IMG_PATH) else None  # 存在则读，否则 None
if img is None:                                          # 若读不到
    img = np.zeros((400, 500, 3), dtype=np.uint8)        # 兜底图
    cv2.rectangle(img, (100, 100), (300, 300), (0, 0, 255), -1)  # 红色实心矩形

img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)            # 供显示用

src = img.copy()                                      # 源图
dst = np.zeros_like(img)                              # 全黑目标图，与源图同形
mask2 = np.zeros(img.shape[:2], dtype=np.uint8)       # 单通道掩码
mask2[100:300, 100:300] = 255                         # 中间矩形置 255

count = int((mask2 == 255).sum())                     # 统计掩码内像素个数
total = mask2.size                                    # 掩码总像素数
print(f'掩码内像素数 = {count}，掩码总像素数 = {total}，占比 = {count}/{total} = {count/total:.2%}')  # 带算式

dst[mask2 == 255] = src[mask2 == 255]                 # 布尔索引：只在掩码区域把源图像素复制到目标图

plt.figure(figsize=(12, 4))                           # 画布
plt.subplot(1, 3, 1); plt.imshow(img_rgb);              plt.title('Source'); plt.axis('off')  # 源图
plt.subplot(1, 3, 2); plt.imshow(mask2, cmap='gray');   plt.title('Mask');   plt.axis('off')  # 掩码
plt.subplot(1, 3, 3); plt.imshow(cv2.cvtColor(dst, cv2.COLOR_BGR2RGB)); plt.title('Copied with mask'); plt.axis('off')  # 结果
plt.tight_layout()                                    # 调整间距
plt.show()                                            # 渲染

print('掩码外像素 dst[0, 0] =', dst[0, 0], ' ← 应为 [0, 0, 0]')  # [0,0,0]
print('掩码内像素 dst[200, 200] =', dst[200, 200], ' 源图 src[200, 200] =', src[200, 200],
      ' → 是否相等:', np.array_equal(dst[200, 200], src[200, 200]))  # 掩码内复制源图

## 1.9.6 图像混合：addWeighted

### 公式

```
dst = src1 × alpha + src2 × beta + gamma
```

- `alpha`、`beta` 是权重（通常和为 1.0）；
- `gamma` 是偏置（默认 0）；
- 常用于两张图融合、透明叠加。

下面把原图和提亮图按 **0.6 / 0.4** 混合，**逐通道打印算式**。

In [ ]:
# ============ 实例：addWeighted 图像混合（本单元自包含） ============
import os                        # 导入 os，用于判断文件是否存在
import numpy as np               # 导入 NumPy，用于 clip 和生成兜底图
import cv2                       # 导入 OpenCV，用于 multiply 与 addWeighted
import matplotlib.pyplot as plt  # 导入 Matplotlib，用于显示

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS', 'DejaVu Sans']  # 中文字体
plt.rcParams['axes.unicode_minus'] = False  # 关闭负号 Unicode 化

# --- 读图：优先真实图，读不到则生成测试图 ---
REAL_IMG_PATH = 'P&V/cat.jpg'                            # 真实图像路径
img = cv2.imread(REAL_IMG_PATH) if os.path.exists(REAL_IMG_PATH) else None  # 存在则读，否则 None
if img is None:                                          # 若读不到
    img = np.zeros((200, 300, 3), dtype=np.uint8)        # 兜底图
    img[:, :, 1] = 120                                   # G 通道为 120

img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)        # 供显示

bright_mul = cv2.multiply(img, 1.5)                   # 先乘 1.5 得到提亮图
bright_mul = np.clip(bright_mul, 0, 255).astype(np.uint8)  # clip 到 0~255 并转 uint8

alpha, beta, gamma = 0.6, 0.4, 0                    # 权重与偏置
blend = cv2.addWeighted(img, alpha, bright_mul, beta, gamma)  # 按公式混合

# --- 逐通道打印推导 ---
pix_o = img[0, 0]                                     # 原图像素
pix_b = bright_mul[0, 0]                              # 提亮像素
pix_m = blend[0, 0]                                   # 混合结果
print(f'公式：dst = src1 × {alpha} + src2 × {beta} + {gamma}')
print(f'以像素 img[0,0] 为例，原图 (B={pix_o[0]}, G={pix_o[1]}, R={pix_o[2]})，提亮图 (B={pix_b[0]}, G={pix_b[1]}, R={pix_b[2]})：')
for c, name in enumerate(['B', 'G', 'R']):             # 遍历三通道
    v1, v2 = int(pix_o[c]), int(pix_b[c])              # 原值
    s = v1 * alpha + v2 * beta + gamma                 # 数学结果
    print(f'  {name} 通道: {v1} × {alpha} + {v2} × {beta} + {gamma} = {s:.2f}  → 取整 = {int(pix_m[c])}')

plt.figure(figsize=(15, 4))                           # 画布
plt.subplot(1, 3, 1); plt.imshow(img_rgb);            plt.title('Original'); plt.axis('off')  # 原图
plt.subplot(1, 3, 2); plt.imshow(cv2.cvtColor(bright_mul, cv2.COLOR_BGR2RGB)); plt.title('Bright'); plt.axis('off')  # 提亮
plt.subplot(1, 3, 3); plt.imshow(cv2.cvtColor(blend, cv2.COLOR_BGR2RGB)); plt.title(f'Blend {alpha}/{beta}'); plt.axis('off')  # 混合
plt.tight_layout()                                    # 调整间距
plt.show()                                            # 渲染

---
# 1.10 性能基础

## 1.10.0 为什么单独讲一节“性能”

前面所有操作都假设“图很小、跑一次就完”。但真实项目里：

- **视频**：每秒 30 帧，一帧 1920×1080，一分钟就是 1800 帧；
- **批处理**：几万张图依次处理；
- **嵌入式**：算力有限，同一个算法慢 10 倍可能就跑不动。

所以“能不能跑”只是第一步，**“跑得快不快”才是工程上真正决定方案能否落地的关键**。

### 本节讲三个层次

| 层次 | 手段 | 效果 |
|------|------|------|
| 1. 写法层 | 向量化、避免 Python 循环 | 最大，几十到几百倍 |
| 2. 库开关层 | `cv2.setUseOptimized`、多线程 | 中，几倍 |
| 3. 硬件层 | `cv2.UMat` / OpenCL / GPU | 视硬件，可能几倍到几十倍 |

**顺序很重要**：先把写法写对，再谈开关和硬件；否则开了 OpenCL 也快不了。

## 1.10.1 层次一：向量化优先（实测对比）

**核心原则**：能用 NumPy / OpenCV 向量化，就不要写 Python 循环。

**为什么？**

- Python 循环每次迭代都要做类型检查、对象分派，开销极大；
- NumPy / OpenCV 的向量化操作走的是 C 循环，几乎没有 Python 开销；
- 逐像素访问可能慢 **几十到几百倍**。

下面用同一件事（复制一张 512×512 图）实测两种写法，**打印耗时和倍率**。

In [ ]:
# ============ 实例：Python 循环 vs 向量化（本单元自包含） ============
import time                      # 导入 time，用于计时
import numpy as np               # 导入 NumPy，用于构造随机图和向量化拷贝

big = np.random.randint(0, 256, (512, 512, 3), dtype=np.uint8)  # 构造 512×512×3 随机图
print(f'测试规模：{big.shape}，总像素数 = 512 × 512 = {512*512}')  # 打印规模，带算式

# 方法一：Python 双重循环
t0 = time.perf_counter()                              # 开始计时
out1 = np.zeros_like(big)                             # 预分配输出数组
h, w = big.shape[:2]                                  # 取高、宽
for y in range(h):                                    # 逐行遍历
    for x in range(w):                                # 逐列遍历
        out1[y, x] = big[y, x]                        # 逐像素复制
t1 = time.perf_counter()                              # 结束计时
t_loop = t1 - t0                                      # 记录耗时
print('Python 循环耗时: %.4f 秒' % t_loop)            # 打印耗时

# 方法二：向量化
t0 = time.perf_counter()                              # 开始计时
out2 = big.copy()                                     # 底层 C 循环整块内存复制
t1 = time.perf_counter()                              # 结束计时
t_vec = t1 - t0                                       # 记录耗时
print('向量化耗时     : %.4f 秒' % t_vec)            # 打印耗时

print('两种结果是否一致:', np.array_equal(out1, out2))  # True
if t_vec > 0:
    print(f'向量化比 Python 循环快约 {t_loop / t_vec:.1f} 倍（{t_loop:.4f} / {t_vec:.4f}）')  # 倍率带算式

**结论**：向量化比 Python 循环快 **1~2 个数量级**，且结果一致。

> **能写成 `img + 1`，就不要写 `for`。**

## 1.10.2 层次二：`cv2.setUseOptimized`（优化开关）

### 它是什么

`cv2.setUseOptimized(True/False)` 用来**开关 OpenCV 内部的优化实现**。

### 优化指的是什么

OpenCV 很多函数有**两套实现**：

- **通用版**：纯 C/C++ 写的、任何平台都能跑的实现；
- **优化版**：针对特定 CPU 指令集（SSE / AVX / NEON）、IPP 等做的加速版本。

`setUseOptimized(True)` 就是告诉 OpenCV：**能用优化版就用优化版**。

### 默认是开的吗

- **默认 `True`**（绝大多数官方发行版）；
- 你可能根本不需要手动设，但**显式设一遍没坏处**；
- 用 `cv2.useOptimized()` 查询当前状态。

### 关了会怎样

- 同一算法可能慢 **几倍**；
- 结果**一般不变**，只是速度慢；
- 特殊情况下（调试、定位 bug）会临时关掉。

### 顺带讲：线程数

- `cv2.getNumThreads()` / `cv2.setNumThreads(n)`：OpenCV 内部并行线程数；
- 默认是 CPU 核数；
- 单张小图多线程不一定快，因为线程调度有开销；
- 视频批处理、大图处理时多线程有效。

下面实测：**开 / 关优化，对比同一操作的耗时**。

In [ ]:
# ============ 实例：setUseOptimized 开 / 关对比（本单元自包含） ============
import os                        # 导入 os，用于判断文件是否存在
import numpy as np               # 导入 NumPy，用于生成兜底图
import cv2                       # 导入 OpenCV，用于测试优化开关

# --- 读图：优先真实图，读不到则生成测试图 ---
REAL_IMG_PATH = 'P&V/cat.jpg'                            # 真实图像路径
img = cv2.imread(REAL_IMG_PATH) if os.path.exists(REAL_IMG_PATH) else None  # 存在则读，否则 None
if img is None:                                          # 若读不到
    img = np.random.randint(0, 256, (400, 500, 3), dtype=np.uint8)  # 随机兜底图

print('当前优化状态:', cv2.useOptimized(), ' ← 默认 True')  # 查询当前优化开关状态
print('线程数      :', cv2.getNumThreads())                # 查询当前并行线程数
print()                                                    # 空行

# --- 先测：开优化时的耗时 ---
cv2.setUseOptimized(True)                              # 显式开启优化
t0 = cv2.getTickCount()                                # 记录开始时钟计数
for _ in range(100):                                   # 执行 100 次高斯模糊
    cv2.GaussianBlur(img, (15, 15), 0)                 # 高斯模糊
t1 = cv2.getTickCount()                                # 记录结束时钟计数
freq = cv2.getTickFrequency()                          # 每秒时钟计数
t_on = (t1 - t0) / freq                                # 把计数差转换为秒
print(f'优化 ON  : 100 次高斯模糊耗时 = ({t1} - {t0}) / {freq:.0f} = {t_on:.4f} 秒')  # 打印耗时带算式

# --- 再测：关优化时的耗时 ---
cv2.setUseOptimized(False)                             # 关闭优化
t0 = cv2.getTickCount()                                # 开始计数
for _ in range(100):                                   # 同样执行 100 次
    cv2.GaussianBlur(img, (15, 15), 0)                 # 高斯模糊
t1 = cv2.getTickCount()                                # 结束计数
t_off = (t1 - t0) / freq                               # 换算成秒
print(f'优化 OFF : 100 次高斯模糊耗时 = ({t1} - {t0}) / {freq:.0f} = {t_off:.4f} 秒')  # 打印耗时带算式

# --- 恢复默认（开），并打印对比 ---
cv2.setUseOptimized(True)                              # 恢复默认开启
print()                                                # 空行
print('优化状态已恢复:', cv2.useOptimized())           # 确认已恢复
if t_on > 0:                                           # 避免除零
    print(f'关闭优化后慢了约 {t_off / t_on:.2f} 倍（{t_off:.4f} / {t_on:.4f}）')  # 倍率带算式

**结论**：

- 优化开关**默认开**，手动设一遍没坏处；
- 关掉后同一操作会慢若干倍（具体倍数看 CPU 和算法）；
- 这个开关**只影响 OpenCV 内部实现**，不影响你写的 Python 代码；
- **先保证写法和向量化，再谈这个开关**。

## 1.10.3 计时：`cv2.getTickCount` 与 `time.perf_counter`

**为什么要讲计时？** 谈性能不能凭感觉，要**用数字说话**。

两个常用工具：

| 工具 | 位置 | 适合测 |
|------|------|--------|
| `cv2.getTickCount` | OpenCV C 层 | OpenCV 函数（`GaussianBlur` 等） |
| `time.perf_counter` | Python 层 | NumPy / Python 代码 |

**注意**：

- `cv2.getTickCount` 返回的是**时钟计数**，要除以 `cv2.getTickFrequency()` 才是秒；
- `time.perf_counter` 直接返回秒；
- 两者都建议**跑多次取平均**，因为单次抖动大。

In [ ]:
# ============ 实例：两种计时方式（本单元自包含） ============
import time                      # 导入 time，用于 perf_counter 计时
import os                        # 导入 os，用于判断文件是否存在
import numpy as np               # 导入 NumPy，用于生成兜底图
import cv2                       # 导入 OpenCV，用于 getTickCount 计时

# --- 读图：优先真实图，读不到则生成测试图 ---
REAL_IMG_PATH = 'P&V/cat.jpg'                            # 真实图像路径
img = cv2.imread(REAL_IMG_PATH) if os.path.exists(REAL_IMG_PATH) else None  # 存在则读，否则 None
if img is None:                                          # 若读不到
    img = np.random.randint(0, 256, (400, 500, 3), dtype=np.uint8)  # 随机兜底图

t0 = cv2.getTickCount()                                # OpenCV 计时开始
for _ in range(100):                                   # 跑 100 次
    cv2.GaussianBlur(img, (15, 15), 0)                 # 高斯模糊
t1 = cv2.getTickCount()                                # OpenCV 计时结束
freq = cv2.getTickFrequency()                          # 时钟频率
print(f'cv2.getTickCount 计时: ({t1} - {t0}) / {freq:.0f} = {(t1 - t0) / freq:.4f} 秒')  # 带算式

t0 = time.perf_counter()                               # Python 计时开始
for _ in range(100):                                   # 跑 100 次
    _ = img + 1                                        # NumPy 整体加 1（向量化）
t1 = time.perf_counter()                               # Python 计时结束
print(f'time.perf_counter 计时 : {t1:.4f} - {t0:.4f} = {t1 - t0:.4f} 秒')  # 带算式

## 1.10.4 层次三：`UMat` 与 OpenCL 加速初步

### 为什么需要它

前面两种手段都在 **CPU** 上。如果机器有 **OpenCL 设备（核显 / 独显 / 某些 SoC）**，
可以把图像放到设备上算，进一步加速。

### 是什么

- `cv2.UMat`：OpenCV 的**透明 API 容器**，放在设备（OpenCL）上的“图像句柄”；
- 用法：`cv2.GaussianBlur(cv2.UMat(img), ...)`；
- 返回的也是 `UMat`，需要 `.get()` 取回 NumPy 数组；
- **对代码是透明的**：写法几乎不变，性能由 OpenCV 内部决定。

### 怎么检测 / 开关

- `cv2.ocl.haveOpenCL()`：当前 OpenCV 是否支持 OpenCL；
- `cv2.ocl.setUseOpenCL(True)`：开启；
- `cv2.ocl.useOpenCL()`：查询。

### 什么时候不要用

- **小图**：数据上传 / 下载开销大于收益；
- **没有 OpenCL 设备**：自动退化为 CPU，没意义；
- **纯 NumPy 运算**：UMat 只对 OpenCV 函数有效。

In [ ]:
# ============ 实例：UMat 与 OpenCL（本单元自包含） ============
import os                        # 导入 os，用于判断文件是否存在
import numpy as np               # 导入 NumPy，用于生成兜底图
import cv2                       # 导入 OpenCV，用于 OpenCL 与 UMat
import matplotlib.pyplot as plt  # 导入 Matplotlib，用于显示

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS', 'DejaVu Sans']  # 中文字体
plt.rcParams['axes.unicode_minus'] = False  # 关闭负号 Unicode 化

# --- 读图：优先真实图，读不到则生成测试图 ---
REAL_IMG_PATH = 'P&V/cat.jpg'                            # 真实图像路径
img = cv2.imread(REAL_IMG_PATH) if os.path.exists(REAL_IMG_PATH) else None  # 存在则读，否则 None
if img is None:                                          # 若读不到
    img = np.zeros((300, 400, 3), dtype=np.uint8)        # 兜底图
    cv2.rectangle(img, (80, 80), (320, 220), (0, 0, 255), -1)  # 红色实心矩形

img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)        # 供显示用

print('OpenCL 可用:', cv2.ocl.haveOpenCL(), ' ← True 表示当前 OpenCV 支持 OpenCL')  # 是否支持 OpenCL
if cv2.ocl.haveOpenCL():                              # 若支持
    cv2.ocl.setUseOpenCL(True)                        # 开启 OpenCL
    print('OpenCL 已开启:', cv2.ocl.useOpenCL())      # 确认已启用
else:                                                 # 若不支持
    print('OpenCL 不可用，跳过 UMat 演示')             # 提示跳过

try:                                                  # 用 try 保护，以防 UMat 不可用
    u = cv2.UMat(img)                                 # 把 NumPy 图像包装成 UMat
    u_blur = cv2.GaussianBlur(u, (15, 15), 0)         # 在 UMat 上做高斯模糊
    blur_np = u_blur.get()                            # 把 UMat 取回成 NumPy 数组
    print('UMat 高斯模糊后 shape:', blur_np.shape, ' 原图 shape:', img.shape, ' → 一致')  # 打印结果形状

    plt.figure(figsize=(10, 4))                       # 画布
    plt.subplot(1, 2, 1); plt.imshow(img_rgb);        plt.title('Original'); plt.axis('off')  # 原图
    plt.subplot(1, 2, 2); plt.imshow(cv2.cvtColor(blur_np, cv2.COLOR_BGR2RGB)); plt.title('UMat GaussianBlur'); plt.axis('off')  # 结果
    plt.tight_layout()                                # 调整间距
    plt.show()                                        # 渲染
except Exception as e:                                # 捕获异常
    print('UMat 演示失败:', e)                        # 打印错误

## 1.10.5 其他性能建议（一句话清单）

- **写法优先**：向量化、避免 Python 循环；
- **原地操作**：`cv2.add(src, val, dst=src)`，减少内存分配；
- **少用 `cv2.split` / `cv2.merge`**：能索引通道就索引；
- **大图先缩小**：能降采样就降采样，再处理；
- **视频逐帧处理**：不要一次性把整个视频读进内存；
- **计时用数字说话**：`cv2.getTickCount` / `time.perf_counter`；
- **有 GPU 用 GPU**：`cv2.UMat` + OpenCL；没 GPU 别硬上。

---
# 1.11 小结

- OpenCV 是传统 CV 工具箱，C++ 底层，Python 只是封装；
- 安装优先 `pip install opencv-python`，contrib 用 `opencv-contrib-python`，服务器用 `headless`；
- **Anaconda 新版默认不带 OpenCV**，需自己装；
- 图像 = `ndarray`，`shape=(H,W,C)`，`dtype=uint8`，通道 **BGR**；
- **讲图像表示必须把图显示出来**：真实图 + ROI + 缩小 + 手动小数组对照；
- 视图 vs 副本、内存布局/步长、8位/16位/32F 差异；
- 色彩空间：Gray / BGR / RGB / HSV / YCrCb / Lab，用 `cv2.cvtColor`；
- 读 `imread`、显示 `imshow` + `waitKey` + `destroyAllWindows`、保存 `imwrite`；
- EXIF 用 Pillow 读取；
- 视频用 `VideoCapture` / `VideoWriter`，必须 `release()`；
- 视频属性 `get/set(CAP_PROP_*)`，图像序列输入输出；
- 像素 `img[y, x]`、ROI 切片、通道 `split` / `merge`；
- 算术 `cv2.add`（饱和截断）、`cv2.multiply` / `cv2.divide`，位运算做掩码，`copyTo` 用 NumPy 等价写法，`addWeighted` 混合；
- 性能：**向量化 → `setUseOptimized` → `UMat` / OpenCL** 三个层次，**先写法后硬件**；
- 计时：`cv2.getTickCount`、`time.perf_counter`；
- 常见坑：中文路径、BGR/RGB、版本 API 变化、`imread` 返回 `None`、Anaconda 默认不带 OpenCV。

**核心原则**：每一处讲解都配可运行代码，每一段代码都显示结果；**每个代码单元都相对独立、可单独运行；每一行代码都有逐行注释；涉及数值运算的地方先输出算式再输出结果**。

# 附录：常见问题速查

| 问题 | 可能原因 | 解决 |
|------|----------|------|
| `import cv2` 报错 | 未安装 / 环境错 | `pip install opencv-python` |
| `imread` 返回 None | 路径错 / 中文路径 / 文件损坏 | 检查路径，用 `imdecode` |
| 颜色偏红蓝 | BGR/RGB 混用 | `cv2.cvtColor(BGR2RGB)` |
| 窗口一闪而过 | 忘记 `waitKey` | 加 `cv2.waitKey(0)` |
| Jupyter 弹不出窗口 | 无 GUI / 后端冲突 | 用 Matplotlib 或 headless |
| SIFT 找不到 | contrib 未安装 | 装 `opencv-contrib-python` |
| `findContours` 返回值不对 | 版本差异 | 4.x 返回 2 个值 |
| 中文路径读写失败 | OpenCV 限制 | `imdecode` / `imencode` + `tofile` |
| 摄像头打不开 | 被占用 / 无权限 | 换索引 / 检查驱动 |
| 处理速度慢 | Python 循环 | 向量化、`cv2` 内置函数 |
| Anaconda 装了却没 cv2 | 新版默认不带 | `conda install -c conda-forge opencv` |
| 图像显示全白/全黑 | dtype / 范围不对 | float 归一化到 0~1，或转 uint8 |
| ROI 改了原图也变 | 视图不是副本 | 需要独立副本用 `.copy()` |
| `cv2` 报内存不连续 | ROI 切片不连续 | `np.ascontiguousarray` |
| 想用 GPU 加速 | 没用 UMat/OpenCL | `cv2.UMat` + `cv2.ocl.setUseOpenCL(True)` |
| 加法结果不符合预期 | NumPy `+` 溢出绕回，不是饱和 | 需要饱和用 `cv2.add` |

---
# 附录

本附录按 **正文中出现的顺序**，对正文提到的关键概念做补充讲解。

正文中 SIFT 出现在：

- **1.1.2 能力边界**：把 SIFT 列为 OpenCV 支持的特征提取算法之一；
- **1.2.2 三个 pip 包的区别**：SIFT 属于 contrib 模块；
- **1.3.2 坑 1：SIFT 在 contrib**：用 `cv2.SIFT_create()` 验证 contrib 是否可用。

因此附录 A 按「它是什么 → 为什么需要它 → 它怎么工作 → 在 OpenCV 中怎么用、怎么避坑」的顺序展开。

## 附录 A：SIFT 深入讲解

### A.1 SIFT 是什么

SIFT，全称 **Scale-Invariant Feature Transform**（尺度不变特征变换），
是计算机视觉中一个非常经典的 **特征提取算法**。

它要解决的问题是：

> **无论图像怎么缩放、旋转，甚至改变光照，都能稳定地找到同一批“关键点”，并生成可以互相匹配的描述子。**

它在正文 1.1.2 中被列为 OpenCV 支持的特征提取算法之一，
与 ORB、AKAZE 并列。

### A.2 为什么需要 SIFT

在 SIFT 之前，Harris 角点检测是最常用的特征点检测方法，但它有一个明显的短板：

> **不具备尺度不变性。**

想象一个角落：

- 在小图里，它是一个尖角；
- 你把图放大后，这个角落可能变得平缓，不再是角点。

这就导致：**同一张图缩放后，Harris 检测到的角点集合会变**，无法稳定匹配。

SIFT 的核心突破，就是让特征点**即使在图像缩放后也能被稳定检测出来**。

### A.3 SIFT 的四步核心流程

SIFT 算法主要由四个步骤组成，每一步都在为“稳定性”服务。

#### A.3.1 尺度空间极值检测

这是 SIFT 的灵魂。

算法会构建一个“图像金字塔”，对图像做不同尺度的高斯模糊，
然后通过比较相邻尺度的图像（**高斯差分**，DoG，Difference of Gaussian），
来寻找那些在**位置和尺度上**都稳定的极值点。

> 这就像你同时用不同大小的放大镜去看图片，
> 找到那些在多个放大倍数下都“显眼”的点。

#### A.3.2 关键点定位

初步找到的点还不够精确。
这一步会通过拟合三维二次函数来精确定位关键点的位置和尺度，
同时 **剔除低对比度的点和边缘响应点**。

边缘上的点容易滑动，不够稳定，所以要过滤掉。

#### A.3.3 方向分配

为了让特征点不受图像旋转的影响，
算法会根据关键点邻域内像素的**梯度方向**，给每个关键点分配一个**主方向**。

之后所有的计算都会相对于这个方向进行，
这样即使图像旋转了，特征描述也保持一致。

#### A.3.4 关键点描述子生成

最后，在关键点周围取一个 **16×16 的窗口**，
分成 16 个小块，计算每个小块内 **8 个方向的梯度直方图**。
合起来就得到一个 **128 维的特征向量**。

这个向量就是该关键点的“指纹”，对光照变化也做了归一化处理，非常鲁棒。

### A.4 SIFT 在 OpenCV 中的使用与“坑”

使用 SIFT 时，最关键的一点是 **版本和安装包的问题**。

#### A.4.1 版本差异

- **老版本（OpenCV 3.x 到 4.4.0 之前）**：
  SIFT 因为专利问题，被移到了 `opencv-contrib-python` 包里。
  你需要 `pip install opencv-contrib-python`，
  并且通过 `cv2.xfeatures2d.SIFT_create()` 来调用。

- **新版本（OpenCV 4.4.0 及以后）**：
  SIFT 专利已过期，它被移回了主仓库。
  现在你只需要安装标准的 `opencv-python`，
  就可以直接通过 **`cv2.SIFT_create()`** 来使用了。

这正是正文 1.2.2 中「SIFT 属于 contrib」和 1.3.2「坑 1：SIFT 在 contrib」
要提醒读者注意的地方。

#### A.4.2 最简单的调用代码

```python
import cv2

# 创建 SIFT 检测器
sift = cv2.SIFT_create()

# 检测关键点和计算描述子
keypoints, descriptors = sift.detectAndCompute(gray_image, None)
```

其中：

- `keypoints` 是一个包含位置、尺度、方向等信息的列表；
- `descriptors` 就是每个关键点对应的 **128 维描述子向量**。

#### A.4.3 完整的可运行示例

下面这段代码可以直接在 Notebook 中运行：
它读一张图、检测 SIFT 关键点、把关键点画在图上并显示。

In [ ]:
# ============ 附录 A 实例：SIFT 检测 + 可视化（本单元自包含） ============
import os                        # 导入 os，用于判断文件是否存在
import numpy as np               # 导入 NumPy，用于生成兜底图
import cv2                       # 导入 OpenCV，用于 SIFT 检测和绘图
import matplotlib.pyplot as plt  # 导入 Matplotlib，用于显示

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS', 'DejaVu Sans']  # 中文字体
plt.rcParams['axes.unicode_minus'] = False  # 关闭负号 Unicode 化

# --- 读图：优先真实图，读不到则生成测试图 ---
REAL_IMG_PATH = 'P&V/cat.jpg'                            # 真实图像路径
img = cv2.imread(REAL_IMG_PATH) if os.path.exists(REAL_IMG_PATH) else None  # 存在则读，否则 None
if img is None:                                          # 若读不到
    img = np.zeros((400, 500, 3), dtype=np.uint8)        # 兜底图
    cv2.rectangle(img, (100, 100), (300, 300), (0, 0, 255), -1)  # 红色实心矩形
    cv2.circle(img, (400, 120), 60, (0, 255, 0), -1)     # 绿色实心圆

gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)    # SIFT 通常在灰度图上检测

try:                                            # 尝试执行可能因为缺 SIFT 而失败的代码
    sift = cv2.SIFT_create()                    # 创建 SIFT 检测器
    kps, des = sift.detectAndCompute(gray, None)  # 检测关键点 + 计算描述子
    print('关键点数量:', len(kps), ' ← 每个关键点有位置、尺度、方向')  # 打印关键点数量
    print('描述子 shape:', des.shape, ' = (关键点数, 128)')  # 每个描述子 128 维

    img_kp = cv2.drawKeypoints(                 # 在图上绘制关键点
        img, kps, None,
        flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)  # 画出带尺度和方向的圆
    img_kp_rgb = cv2.cvtColor(img_kp, cv2.COLOR_BGR2RGB)    # BGR -> RGB

    plt.figure(figsize=(10, 5))                 # 画布
    plt.subplot(1, 2, 1)                        # 左图
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))  # 原图
    plt.title('Original')                       # 标题
    plt.axis('off')                             # 关闭坐标轴
    plt.subplot(1, 2, 2)                        # 右图
    plt.imshow(img_kp_rgb)                      # 带关键点的图
    plt.title(f'SIFT keypoints ({len(kps)})')   # 标题带上关键点数量
    plt.axis('off')                             # 关闭坐标轴
    plt.tight_layout()                          # 调整间距
    plt.show()                                  # 渲染
except AttributeError as e:                     # 若当前 OpenCV 没有 SIFT
    print('当前环境没有 SIFT，请安装 opencv-contrib-python 或升级到 OpenCV 4.4.0+。')
    print('错误信息:', e)                       # 打印错误信息

### A.5 SIFT 与其他特征算法对比

| 算法 | 尺度不变 | 旋转不变 | 速度 | 是否需要 contrib |
|------|----------|----------|------|------------------|
| SIFT | ✅ | ✅ | 慢 | 老版本需要，新版本不需要 |
| SURF | ✅ | ✅ | 中 | 需要 |
| ORB | ✅ | ✅ | 快 | 不需要 |
| AKAZE | ✅ | ✅ | 中 | 不需要 |

**选型建议**：

- 追求**精度**、不在乎速度 → SIFT；
- 追求**速度**、嵌入式 / 实时 → ORB；
- 想要**免费且性能均衡** → AKAZE。

### A.6 一句话总结

> SIFT 是 **尺度不变特征变换**，通过构建尺度空间、精确定位关键点、分配方向、
> 生成 128 维描述子，实现了对缩放、旋转、光照的鲁棒性。
> 在 OpenCV 中，**新版本直接用 `cv2.SIFT_create()`**，
> **老版本需要 `opencv-contrib-python` 和 `cv2.xfeatures2d.SIFT_create()`**。